# Group contrast vs individual deviation -- Pancreatic Cancer (Moore et al.)

Two sample arms, each paired with its own DESeq2 runs and laid out with the same figures in the same order.

| | arm | samples (neoplasm / HC) | DESeq2 runs |
|---|---|---|---|
| **Part A** | Moore `Batch_1` + `Batch_2` + `Batch_3` | 72 / 69 | `pancreatic_deg_pooled`: our own DESeq2 (+ RUVg k=1-3) on the pooled cohort, not a reproduction of Moore et al. |
| **Part B** | Moore `Batch_1` only | 39 / 48 | `pancreatic_deg`: the CEDAR cohort Moore et al. fitted on, same designs |

Each part: Signal Number Heatmap -> UpSet -> then, for the pancreas, liver and Open Targets panels in turn,
cutoff sweep -> marker recovery -> heatmap -> Venn. Samples on the NM side are exactly the samples each
DESeq2 run was fitted on (read back from its `ruvg_W/*.csv`), so the two sides cannot drift.

Hit rules. Tissue marker genes (pancreas, liver panels; `hpa_panels.tct_panel_table`): any detection,
per-sample BH q<0.05 with Z>0, and in a patient >= MIN_HIT_READS (10) reads (HC hits carry no read
condition). Everything else -- genome-wide unions, Open Targets genes, n_sig -- uses two-sided BH q<0.05
and 10% detection. Part C holds the supporting tables and the Moore comparison; Part D the per-patient organ signal.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
import pickle
import sys
from collections import Counter
from pathlib import Path

import gseapy as gp
import h5py
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

os.environ["NM_PIPELINE"] = "p0"
parent_dir = str(Path.cwd().parents[1])
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from viz_style import FS_ANNOT_LG, FS_ANNOT_SM, apply_style
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.notebookscripts import hpa_panels
from MixedEffectsModeling.core.calibration import bh_fdr_reject
%matplotlib inline 

SIG_SOURCE = 'insample'
RUVG_PICK = 'ruvg_k2'
DESIGN_PAIR = ['no_covariate', RUVG_PICK]
DEG_DIR = config.PANCREATIC_DEG_DIR
MATCHED_BATCH = 'Moore et al._Batch_1'
MOORE_BATCHES = [MATCHED_BATCH, 'Moore et al._Batch_2', 'Moore et al._Batch_3']
NEOPLASTIC = ['IPMN', 'Islet Cell Tumor', 'PDAC']
STAGE_C = {'PDAC': '#0072B2', 'IPMN': '#E69F00', 'Islet Cell Tumor': '#009E73'}
DESIGN_C = {'no_covariate': '#A2A2A2', 'ruvg_k1': '#489ACA', 'ruvg_k2': '#009E73',
            'ruvg_k3': '#CC79A7'}
DESIGN_LABEL = {'no_covariate': 'DESeq2\nDefault', 'ruvg_k1': 'DESeq2 +\nRUVg (k=1)',
                'ruvg_k2': 'DESeq2 +\nRUVg (k=2)', 'ruvg_k3': 'DESeq2 +\nRUVg (k=3)'}
CASE_OF = {'PDAC': 'PDAC', 'IPMN': 'IPMN', 'Islet Cell Tumor': 'IsletCellTumor',
           'all neoplasm': 'PancreaticNeoplasm'}
FDR_Q = 0.05
DESEQ2_PADJ = 0.05
N_HC_DRAWS = 20
HC_NULL_PCT = 99      # HC above this percentile of the FULL HC pool are not null material
SEED = 0

OUT = config.GROUP_VS_INDIV_DIR / SIG_SOURCE
FIG = config.GROUP_VS_INDIV_FIG_DIR / SIG_SOURCE
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
print('source:', SIG_SOURCE, '| designs:', DESIGN_PAIR, '| DEG:', DEG_DIR.name, '->', OUT)

In [ ]:
def load_source(src):
    if src == 'insample':
        ZD = config.ZSCORES_MIXED_DIR
        dm = pd.read_csv(ZD / 'sample_meta.csv')
        hm = pd.read_csv(ZD / 'hc_meta.csv')
        dk = dm['ood_keep'].values.astype(bool)
        return dict(genes=np.array(pickle.load(open(ZD / 'gene_names.pkl', 'rb'))),
                    Z_dis=np.load(ZD / 'Z_disease_shash.npy')[dk],
                    dis=dm.loc[dk, 'sample'].values, dis_batch=dm.loc[dk, 'batch'].values,
                    Z_hc=np.load(ZD / 'Z_hc_shash.npy'), hc=hm['sample'].values,
                    hc_batch=hm['batch'].values)
    Zs, nm, bb, gl = [], [], [], None
    for b in MOORE_BATCHES:
        LD = config.LOBO_MIXED_DIR / b.replace(' ', '_')
        info = json.load(open(LD / 'meta.json'))
        k = np.load(LD / 'ood_mask.npy').astype(bool)
        g = np.array(pickle.load(open(LD / 'gene_names.pkl', 'rb')))
        assert gl is None or np.array_equal(gl, g)
        gl = g
        Zs.append(np.load(LD / 'Z_test_shash.npy')[k])
        nm.append(np.array(info['test_names'])[k])
        bb.append(np.repeat(b, int(k.sum())))
    Z, nm, bb = np.vstack(Zs), np.concatenate(nm), np.concatenate(bb)
    ih = np.isin(nm, pd.read_csv(config.ZSCORES_MIXED_DIR / 'hc_meta.csv')['sample'].values)
    return dict(genes=gl, Z_dis=Z[~ih], dis=nm[~ih], dis_batch=bb[~ih],
                Z_hc=Z[ih], hc=nm[ih], hc_batch=bb[ih])


src = load_source(SIG_SOURCE)
genes = src['genes']
gpos = {g: i for i, g in enumerate(genes)}

with h5py.File(config.H5AD_PATH, 'r') as h:
    dec = lambda a: np.array([x.decode('utf-8', 'replace') if isinstance(x, bytes) else str(x) for x in a])
    g = h['obs/Stage/Condition']
    stage_all = pd.Series(pd.Categorical.from_codes(g['codes'][()], dec(g['categories'][()])).astype(str),
                          index=dec(h['obs/_index'][()]))

dis_stage = pd.Series(src['dis']).map(stage_all).values
dis_idx, hc_idx = pd.Index(src['dis']), pd.Index(src['hc'])

# all arm: every Moore Batch_1-3 pancreatic sample the engine scored
a_moore = np.isin(src['dis_batch'], MOORE_BATCHES)
a_neo, a_panc = a_moore & np.isin(dis_stage, NEOPLASTIC), a_moore & (dis_stage == 'Pancreatitis')
a_hc = np.isin(src['hc_batch'], MOORE_BATCHES)
Z_A, names_A, stage_A = src['Z_dis'][a_neo], src['dis'][a_neo], dis_stage[a_neo]
Z_hc_A0, names_hc_A0 = src['Z_hc'][a_hc], src['hc'][a_hc]


def deg_case_samples(case):
    """The exact case+HC sample list the matched DESeq2 run was fitted on."""
    s = pd.read_csv(DEG_DIR / 'ruvg_W' / f'moore_b1__{case}.csv')['sample'].values
    st = pd.Series(s).map(stage_all).values
    return s, st


# matched arm: taken from the DESeq2 run rather than re-derived, so the two sides cannot drift
mt, mt_stage = deg_case_samples('PancreaticNeoplasm')
is_case = np.isin(mt_stage, NEOPLASTIC)
pat_m, hcm = dis_idx.get_indexer(mt[is_case]), hc_idx.get_indexer(mt[~is_case])
assert (pat_m >= 0).all() and (hcm >= 0).all(), 'DESeq2 sample missing from the Z matrices'
Z_neo0, names_neo0, stage_neo0 = src['Z_dis'][pat_m], src['dis'][pat_m], mt_stage[is_case]
Z_hc_m0, names_hc_m0 = src['Z_hc'][hcm], src['hc'][hcm]

print(f'genes={len(genes)}')
print(f'matched arm ({MATCHED_BATCH}, paired with {DEG_DIR.name}): '
      f'neoplasm={len(names_neo0)}  HC={len(names_hc_m0)}')
print(pd.Series(stage_neo0).value_counts().reindex(NEOPLASTIC).to_string())
print(f'all arm (Moore Batch_1-3): neoplasm={len(names_A)}  pancreatitis={int(a_panc.sum())}  '
      f'HC={len(names_hc_A0)}')
print(pd.Series(stage_A).value_counts().reindex(NEOPLASTIC).to_string())


In [ ]:
MIN_HIT_READS = 10
_read_ok = {}

def read_ok(names):
    miss = [s for s in names if s not in _read_ok]
    if miss:
        with h5py.File(config.H5AD_PATH, 'r') as h:
            obs = pd.Index(dec(h['obs/_index'][()]))
            var = dec(h['var/_index'][()])
            col = pd.Index(var).get_indexer(genes)
            assert (col >= 0).all()
            indptr = h['X/indptr'][()]
            for s in miss:
                i = obs.get_loc(s)
                row = np.zeros(len(var))
                row[h['X/indices'][indptr[i]:indptr[i + 1]]] = h['X/data'][indptr[i]:indptr[i + 1]]
                _read_ok[s] = row[col] >= MIN_HIT_READS
    return np.vstack([_read_ok[s] for s in names])


def bh_sig_matrix(Z, q=FDR_Q):
    return np.vstack([bh_fdr_reject(2 * norm.sf(np.abs(Z[i])), q=q) for i in range(Z.shape[0])])

def marker_sig_matrix(Z, q=FDR_Q, names=None):
    S = bh_sig_matrix(Z, q) & (Z > 0)
    return S if names is None else S & read_ok(names)

Z_neo, names_neo, stage_neo = Z_neo0, names_neo0, stage_neo0
Z_hc_qc, names_hc = Z_hc_m0, names_hc_m0
SIG, SIG_HC = bh_sig_matrix(Z_neo), bh_sig_matrix(Z_hc_qc)
SIG_M, SIG_HC_M = marker_sig_matrix(Z_neo, names=names_neo), marker_sig_matrix(Z_hc_qc)

Z_hc_A, names_hc_A = Z_hc_A0, names_hc_A0
SIG_A, SIG_HC_A = bh_sig_matrix(Z_A), bh_sig_matrix(Z_hc_A)
SIG_A_M, SIG_HC_A_M = marker_sig_matrix(Z_A, names=names_A), marker_sig_matrix(Z_hc_A)
print(f'matched arm: patients {len(names_neo)}, HC {len(names_hc)} | '
      f'all arm: patients {len(names_A)}, HC {len(names_hc_A)}')

HC_NSIG_CACHE = OUT / 'hc_pool_nsig.csv'
if HC_NSIG_CACHE.exists():
    hc_pool_nsig = pd.read_csv(HC_NSIG_CACHE)
else:
    hc_pool_nsig = pd.read_csv(config.ZSCORES_MIXED_DIR / 'hc_meta.csv').assign(
        n_sig=bh_sig_matrix(np.load(config.ZSCORES_MIXED_DIR / 'Z_hc_shash.npy')).sum(axis=1))
    hc_pool_nsig.to_csv(HC_NSIG_CACHE, index=False)

HC_NSIG_CUT = float(np.percentile(hc_pool_nsig['n_sig'], HC_NULL_PCT))
nsig_of = hc_pool_nsig.set_index('sample')['n_sig']
null_keep = pd.Series(names_hc).map(nsig_of).values <= HC_NSIG_CUT
Z_hc_null, names_hc_null = Z_hc_qc[null_keep], names_hc[null_keep]
SIG_HC_NULL = SIG_HC[null_keep]   # BH is per sample, so this is a row subset, not a refit
SIG_HC_NULL_M = SIG_HC_M[null_keep]
print(f'HC null pool: {null_keep.sum()}/{len(null_keep)} of the Batch_1 controls '
      f'(p{HC_NULL_PCT} of all {len(hc_pool_nsig)} HC = n_sig > {HC_NSIG_CUT:.0f})')
print('  disqualified:', {s: int(nsig_of[s]) for s in names_hc[~null_keep]})

per_sample = pd.DataFrame(dict(sample=names_A, stage=stage_A, n_sig=SIG_A.sum(axis=1),
                               in_matched=np.isin(names_A, names_neo)))
per_sample.to_csv(OUT / 'per_sample_nsig.csv', index=False)

if SIG_SOURCE == 'insample':
    ref = pd.read_csv(config.ZSCORES_MIXED_DIR / 'sample_bh_sig.csv').set_index('sample')['n_sig']
    chk = per_sample.assign(pipeline=per_sample['sample'].map(ref))
    print(f'BH vs pipeline sample_bh_sig.csv: exact={int((chk.n_sig == chk.pipeline).sum())}/{len(chk)}')

print(per_sample.groupby('stage')['n_sig'].agg(n='size', median='median', mean='mean')
      .reindex(NEOPLASTIC).round(1).to_string())
print(f'matched-arm patients: n={len(names_neo)}, median n_sig={np.median(SIG.sum(axis=1)):.0f}')
print(f'matched-arm HC (n={len(names_hc)}): median n_sig = {np.median(SIG_HC.sum(axis=1)):.0f}, '
      f'mean = {SIG_HC.sum(axis=1).mean():.1f}')
print(f'all-arm HC (n={len(names_hc_A)}): median n_sig = {np.median(SIG_HC_A.sum(axis=1)):.0f}')


## Shared setup -- marker panels, plotting constants, and the group runs

In [ ]:
MARKER_Z = 1.96
PANEL_CAP = 200
HPA_PANC = {'acinar': ('pancreatic acinar', ['pancreatic acinar cells']),
            'islet': ('pancreatic islet', ['pancreatic islet cells']),
            'ductal': ('pancreatic duct', ['pancreatic duct cells'])}

hpa_atlas = hpa_panels.load_atlas()
full_of = {g.split('.')[0]: g for g in gpos}
pt = hpa_panels.tct_panel_table(dict(HPA_PANC.values()), cap=PANEL_CAP, atlas=hpa_atlas)
pt['panel'] = pt['panel'].map({k: lab for lab, (k, _) in HPA_PANC.items()})
n_sets = pt.groupby('ens')['panel'].nunique()
ambiguous = sorted(pt.loc[pt['ens'].map(n_sets) > 1, 'symbol'].unique())
pt = pt[pt['ens'].map(n_sets) == 1]
MARKER_SETS = {lab: sorted(pt.loc[pt['panel'] == lab, 'symbol']) for lab in HPA_PANC}
print(f'HPA 25.1 tissue cell type + body-wide specificity, pancreas panels (cap {PANEL_CAP})')
print(pt.groupby('panel').size().reindex(HPA_PANC).to_string())
print(f'dropped as ambiguous within the panel ({len(ambiguous)}): {ambiguous}')

with h5py.File(config.H5AD_PATH, 'r') as h:
    var_index = dec(h['var/_index'][()])
    g = h['var/GeneName']
    var_sym = pd.Categorical.from_codes(g['codes'][()], dec(g['categories'][()])).astype(str)
sym_to_ens = pd.Series(var_index, index=var_sym)
sym_of = pd.Series(var_sym, index=var_index)

marker_tbl = (pt[pt['ens'].isin(full_of)].assign(ens=lambda d: d['ens'].map(full_of))
              .rename(columns={'panel': 'set'})[['ens', 'set', 'symbol']].reset_index(drop=True))
ens_of = {e: (st, sym) for e, st, sym in marker_tbl.itertuples(index=False)}
print(f'\nengine-modeled ENSG ids on the panel: {len(marker_tbl)} of {len(pt)}')
print(marker_tbl.groupby('set').size().to_string())


In [ ]:
def raw_counts(samples, ens_list):
    with h5py.File(config.H5AD_PATH, 'r') as h:
        obs_index = list(dec(h['obs/_index'][()]))
        vpos = {e: i for i, e in enumerate(dec(h['var/_index'][()]))}
        cols = np.array([vpos[e] for e in ens_list])
        indptr = h['X/indptr'][()]
        out = np.zeros((len(samples), len(cols)))
        lib = np.zeros(len(samples))
        for r, s in enumerate(samples):
            i = obs_index.index(s)
            sl = slice(indptr[i], indptr[i + 1])
            idx, dat = h['X/indices'][sl], h['X/data'][sl]
            lib[r] = dat.sum()
            pos = pd.Series(np.arange(len(idx)), index=idx).reindex(cols).values
            ok = ~pd.isna(pos)
            out[r, ok] = dat[pos[ok].astype(int)]
    return out, lib


ens_list = marker_tbl['ens'].tolist()
cnt, libsize = raw_counts(np.concatenate([names_neo, names_hc]), ens_list)
is_pat = np.arange(len(cnt)) < len(names_neo)
cpm = cnt / libsize[:, None] * 1e6

rows = []
for s in MARKER_SETS:
    c = marker_tbl['set'].values == s
    tot = cnt[:, c].sum(axis=1)
    cs = cpm[:, c].sum(axis=1)
    a, b = int((tot[is_pat] > 0).sum()), int((tot[~is_pat] > 0).sum())
    rows.append(dict(set=s, n_ens=int(c.sum()),
                     pat_detected=f'{a}/{is_pat.sum()}', hc_detected=f'{b}/{(~is_pat).sum()}',
                     pat_reads=int(tot[is_pat].sum()), hc_reads=int(tot[~is_pat].sum()),
                     pat_cpm=cs[is_pat].mean(), hc_cpm=cs[~is_pat].mean()))
detect = pd.DataFrame(rows)
detect.to_csv(OUT / 'marker_raw_detectability.csv', index=False)
print('is the transcript measurable at all? (raw counts, same samples)')
print(detect.round(4).to_string(index=False))

per_gene = pd.DataFrame(dict(
    set=marker_tbl['set'], symbol=marker_tbl['symbol'],
    pat_nz=(cnt[is_pat] > 0).sum(axis=0), hc_nz=(cnt[~is_pat] > 0).sum(axis=0),
    pat_reads=cnt[is_pat].sum(axis=0).astype(int), hc_reads=cnt[~is_pat].sum(axis=0).astype(int)))
per_gene.to_csv(OUT / 'marker_raw_per_gene.csv', index=False)
print('\nzero everywhere:', sorted(per_gene.loc[(per_gene.pat_reads == 0) & (per_gene.hc_reads == 0), 'symbol']))
print('top contributors:', per_gene.nlargest(4, 'pat_reads')[['symbol', 'pat_reads', 'hc_reads']].to_dict('records'))

In [ ]:
MARKER_MIN_DETECT = 1e-9  # tissue marker genes: any detection
DETECT_MIN = 0.10         # everything else

nz_frac = (cnt > 0).mean(axis=0)
marker_tbl['detect_frac'] = nz_frac
marker_tbl['detected'] = nz_frac >= MARKER_MIN_DETECT
marker_tbl.to_csv(OUT / 'marker_panel.csv', index=False)

mcols = np.array([gpos[e] for e in marker_tbl['ens']])
Zm_pat, Zm_hc = Z_neo[:, mcols], Z_hc_qc[:, mcols]            # matched arm (b1), DESeq2-paired
Zm_A, Zm_hc_A = Z_A[:, mcols], Z_hc_A[:, mcols]               # all arm (b1-b3), hit-level panels
zero_col = nz_frac == 0
print('randomized-quantile Z is not defined by data on a gene with no reads:')
print(f'  marker ENSG with zero counts in all {len(cnt)} samples: {int(zero_col.sum())}/{len(nz_frac)} '
      f'({sorted(marker_tbl.loc[zero_col, "symbol"])})')
print(f'  |Z|>{MARKER_Z} rate there: patient {(np.abs(Zm_pat[:, zero_col]) > MARKER_Z).mean():.3f}, '
      f'HC {(np.abs(Zm_hc[:, zero_col]) > MARKER_Z).mean():.3f}  (0.05 = pure noise)')
print(f'  on detected markers: patient {(np.abs(Zm_pat[:, marker_tbl["detected"]]) > MARKER_Z).mean():.3f}, '
      f'HC {(np.abs(Zm_hc[:, marker_tbl["detected"]]) > MARKER_Z).mean():.3f}')
print(f'\nany-hit rate on the full panel is a set-size counter, not a signal '
      f'(HC should sit at 1-0.95^n if every column were noise):')
for s in MARKER_SETS:
    c = marker_tbl['set'].values == s
    print(f'  {s:7s} n={int(c.sum()):2d}  noise expectation {1 - 0.95 ** c.sum():.2f}  '
          f'HC observed {(np.abs(Zm_hc[:, c]) > MARKER_Z).any(axis=1).mean():.2f}  '
          f'patient {(np.abs(Zm_pat[:, c]) > MARKER_Z).any(axis=1).mean():.2f}')
print(f'\nmarkers surviving detect_frac >= {MARKER_MIN_DETECT}: '
      f'{int(marker_tbl["detected"].sum())}/{len(marker_tbl)}')
print(marker_tbl[marker_tbl['detected']][['set', 'symbol', 'detect_frac']]
      .sort_values(['set', 'detect_frac'], ascending=[True, False]).round(3).to_string(index=False))

In [ ]:
# Same figure on the other kind of panel: Open Targets association genes for pancreatic cancer
# (notebook 4's disease-marker recipe), which are risk/association genes rather than cell-type
# markers. 500 genes is noise-prone at RQR, so this one is restricted to the detected panel.
OT_REF = config.DISEASE_REF_DIR / 'Pancreatic Cancer.json'
OT_TOPN = 500
OT_SCORE_FLOOR = 0.01

ot = json.load(open(OT_REF))
ot_scores = dict(ot['genes'])
ot_syms = set([g for g, s in sorted(ot['genes'], key=lambda gs: -gs[1])
               if s >= OT_SCORE_FLOOR][:OT_TOPN])
sym_g = np.array([sym_of.get(g, g.split('.')[0]) for g in genes])
ot_idx = np.where(np.isin(sym_g, list(ot_syms)))[0]
ot_tbl = pd.DataFrame(dict(ens=genes[ot_idx], symbol=sym_g[ot_idx],
                           score=[ot_scores.get(s, 0.0) for s in sym_g[ot_idx]]))
ot_cnt, _ = raw_counts(np.concatenate([names_neo, names_hc]), ot_tbl['ens'].tolist())
ot_tbl['detect_frac'] = (ot_cnt > 0).mean(axis=0)
ot_tbl['detected'] = ot_tbl['detect_frac'] >= DETECT_MIN
ot_tbl.to_csv(OUT / 'ot_marker_panel.csv', index=False)
print(f"OT '{ot['ot_disease']}' ({ot['efo']}): {len(ot_syms)} symbols -> {len(ot_tbl)} genes, "
      f"{int(ot_tbl['detected'].sum())} at detect_frac >= {DETECT_MIN}")


In [ ]:
SET_C = {'acinar': '#B2182B', 'islet': '#EF8A62', 'ductal': '#67001F'}
Q_TIERS = [0.05, 0.10, 0.20]
Q_GRID = [0.01, 0.05, 0.10, 0.20, 0.30]
N_SWEEP_DRAWS = 2000  # panel-sized draws are cheap; enough for an empirical p
TOP_ROWS = 30  # row cap: more markers get a hit than fit as readable labels
BAR_W, BAR_GAP = 0.9, 1.5 
LABELS = {
    'grid_x': '{n_pat_hit}/{n_pat} Patients carrying hit',
    'grid_y': '{n_rows}/{n_markers_hit} Markers',
    'bar_x': ('patients\nwith a hit', 'markers\nwith a hit'),
    'bar_n': 'n = {n}',
    'sweep_x': 'BH q / DESeq2 padj cutoff',
    'sweep_y': '{ylab} (of {n_panel})',
}


In [ ]:
gene_set = set(genes)
deg, padj_of = {}, {}
for f in sorted(DEG_DIR.glob('moore_b1__*/*.csv.gz')):
    case = f.parent.name.split('__')[1]
    design = f.name.split('.')[0]
    r = pd.read_csv(f, index_col=0)
    deg[(case, design)] = set(r.index[r['padj'].fillna(1.0) < DESEQ2_PADJ]) & gene_set
    padj_of[(case, design)] = r[['padj', 'pvalue', 'log2FoldChange']]

deg_summary = (pd.DataFrame([dict(case=c, design=d, n_sig=len(v)) for (c, d), v in deg.items()])
               .pivot(index='case', columns='design', values='n_sig'))
deg_summary.to_csv(OUT / 'deseq2_nsig_by_case_design.csv')
print(deg_summary.to_string())
print(f'\n{len(deg)} group runs loaded (5 case definitions x 4 designs)')
print('all-run union (reference only, includes Pancreatitis contrasts):',
      len(set().union(*deg.values())))

In [ ]:
# Moore et al.'s 29 group-level markers were liver and immune-tissue genes, not pancreas
# parenchyma: in CEDAR the commonest metastatic sites were liver (6/20) and lymph node (4/20),
# and the immune-tissue genes were read as low tumour-infiltrating immune content. So the two
# tissue axes that study reported are built here from the same HPA recipe as the pancreas
# panel, and laid next to the per-sample metadata.

HPA_TISSUE = {'liver': {'hepatocyte': ['hepatocytes'], 'kupffer': ['kupffer cells'],
                        'hepatic stellate': ['hepatic stellate cells'], 'cholangiocyte': ['cholangiocytes']},
              'lymphoid': {'plasma cell': ['plasma cells'], 'T cell': ['t-cells'],
                           'B / NK cell': ['b-cells', 'nk-cells', 'innate lymphoid cells']}}
META_COLS = ['Met_Site', 'Tumor_Stage', 'Outcome', 'AGE', 'sex']

tissue_tbl = pd.concat([hpa_panels.tct_panel_table(p, cap=PANEL_CAP, atlas=hpa_atlas).assign(panel=t)
                        for t, p in HPA_TISSUE.items()])
tissue_tbl = (tissue_tbl[tissue_tbl['ens'].isin(full_of)].assign(ens=lambda d: d['ens'].map(full_of))
              .rename(columns={'panel': 'set'})[['ens', 'set', 'symbol']].drop_duplicates('ens'))
# the pancreas panel joins in unchanged so all three tissues are scored side by side
tissue_tbl = pd.concat([marker_tbl[['ens', 'symbol']].assign(set='pancreas'), tissue_tbl],
                       ignore_index=True).drop_duplicates('ens')

t_cnt, _ = raw_counts(np.concatenate([names_A, names_hc_A]), tissue_tbl['ens'].tolist())
tissue_tbl['detect_frac'] = (t_cnt > 0).mean(axis=0)
tissue_tbl['detected'] = tissue_tbl['detect_frac'] >= MARKER_MIN_DETECT
tissue_tbl.to_csv(OUT / 'tissue_marker_panel.csv', index=False)
TISSUES = ['pancreas', 'liver', 'lymphoid']
print(f'HPA tissue panels (tissue cell type + body-wide specificity, cap {PANEL_CAP} per cell type)')
print(tissue_tbl.groupby('set').agg(n_genes=('ens', 'size'), detected=('detected', 'sum'))
      .reindex(TISSUES).to_string())


LIVER_C = '#6A3D9A'
liver_tbl = tissue_tbl[tissue_tbl['set'] == 'liver'].reset_index(drop=True)
liver_tbl.to_csv(OUT / 'liver_marker_panel.csv', index=False)

In [ ]:
# One record per arm: the samples a DESeq2 run was fitted on, their Z and hit matrices, the HC null pool
# (HC above the p99 of the full HC pool's n_sig are not null material), and that arm's DESeq2 results.
def build_arm(deg_dir, label):
    s = pd.read_csv(deg_dir / 'ruvg_W' / 'moore_b1__PancreaticNeoplasm.csv')['sample'].values
    st = pd.Series(s).map(stage_all).values
    case = np.isin(st, NEOPLASTIC)
    ip, ih = dis_idx.get_indexer(s[case]), hc_idx.get_indexer(s[~case])
    assert (ip >= 0).all() and (ih >= 0).all(), f'{label}: DESeq2 sample missing from the Z matrices'
    a = dict(label=label, Z=src['Z_dis'][ip], names=src['dis'][ip], stage=st[case],
             Z_hc=src['Z_hc'][ih], names_hc=src['hc'][ih])
    a['SIG'], a['SIG_HC'] = bh_sig_matrix(a['Z']), bh_sig_matrix(a['Z_hc'])
    a['SIG_M'], a['SIG_HC_M'] = marker_sig_matrix(a['Z'], names=a['names']), marker_sig_matrix(a['Z_hc'])
    null = pd.Series(a['names_hc']).map(nsig_of).values <= HC_NSIG_CUT
    a['Z_null'], a['SIG_NULL'], a['SIG_NULL_M'] = a['Z_hc'][null], a['SIG_HC'][null], a['SIG_HC_M'][null]
    a['deg'], a['padj'] = {}, {}
    for f in sorted(deg_dir.glob('moore_b1__*/*.csv.gz')):
        c, d = f.parent.name.split('__')[1], f.name.split('.')[0]
        r = pd.read_csv(f, index_col=0)
        a['deg'][(c, d)] = set(r.index[r['padj'].fillna(1.0) < DESEQ2_PADJ]) & gene_set
        a['padj'][(c, d)] = r[['padj', 'pvalue', 'log2FoldChange']]
    a['n_case'] = {c: int((~np.isin(pd.read_csv(deg_dir / 'ruvg_W' / f'moore_b1__{c}.csv')['sample'], src['hc'])).sum())
                   for c in {c for c, _ in a['deg']}}
    allres = pd.concat(a['padj'].values())
    a['best_padj'] = allres.groupby(level=0)['padj'].min()
    a['best_lfc'] = allres.groupby(level=0)['log2FoldChange'].median()
    print(f"{label}: neoplasm {len(a['names'])} {dict(pd.Series(a['stage']).value_counts())}, "
          f"HC {len(a['names_hc'])} (null pool {int(null.sum())}), {len(a['deg'])} DESeq2 runs from {deg_dir.name}")
    return a


ARMS = {'A': build_arm(config.PANCREATIC_DEG_POOLED_DIR, 'Moore Batch_1-3'),
        'B': build_arm(config.PANCREATIC_DEG_DIR, 'Moore Batch_1')}

In [ ]:
from matplotlib.patches import Ellipse


# Genome-wide figures, one call per arm: Signal Number Heatmap (DESeq2 genes per case x design) and the
# UpSet of the four DESeq2 runs against the per-patient NM union, with the HC-union null as an inset.
ORDER = ['PDAC', 'IPMN', 'IsletCellTumor', 'PancreaticNeoplasm', 'Pancreatitis']
ORDER_LAB = ['PDAC', 'IPMN', 'Islet', 'Neoplasm', 'Pancreatitis']
DESIGN_ORDER = ['no_covariate', 'ruvg_k1', 'ruvg_k2', 'ruvg_k3']
UPSET_TOP = 12
VGROUPS = ['PDAC', 'IPMN', 'all neoplasm']  # islet cell tumour is n=3 in the Batch_1 arm
DESIGN_NAME = {'no_covariate': 'DESeq2 Default', 'ruvg_k1': 'DESeq2 + RUVg (k=1)',
               'ruvg_k2': 'DESeq2 + RUVg (k=2)', 'ruvg_k3': 'DESeq2 + RUVg (k=3)'}
SET_COLOR = {'DESeq2 Default': '#A2A2A2', 'DESeq2 + RUVg (k=1)': '#489ACA',
             'DESeq2 + RUVg (k=2)': '#009E73', 'DESeq2 + RUVg (k=3)': '#CC79A7',
             'Normative Modeling (Union)': '#B2182B'}


def arm_mask(a, grp):
    return np.ones(len(a['names']), bool) if grp == 'all neoplasm' else (a['stage'] == grp)


def arm_hc_draws(a, n, cols=None, M=None, k=N_HC_DRAWS, seed=SEED):
    M = a['SIG_NULL'] if M is None else M
    M = M if cols is None else M[:, cols]
    r = np.random.default_rng(seed)
    ns = min(n, M.shape[0])
    return ns, np.array([int(M[r.choice(M.shape[0], ns, replace=False)].any(axis=0).sum()) for _ in range(k)])


def signal_number_heatmap(a, p):
    t = (pd.DataFrame([dict(case=c, design=d, n=len(v)) for (c, d), v in a['deg'].items()])
         .pivot(index='case', columns='design', values='n').loc[ORDER, DESIGN_ORDER])
    t.to_csv(OUT / f'{p}_deseq2_nsig_by_case_design.csv')
    fig, ax = plt.subplots(figsize=(3.6, 2.12))
    ax.imshow(t.values, cmap='Greys', vmin=0, vmax=t.values.max(), aspect='auto')
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            v = t.values[i, j]
            ax.text(j, i, v, ha='center', va='center', color='white' if v > t.values.max() * 0.55 else 'black')
    ax.set_xticks(range(4), ['none', 'k=1', 'k=2', 'k=3'])
    ax.set_yticks(range(5), [f'{lab} (n={a["n_case"][c]})' for lab, c in zip(ORDER_LAB, ORDER)])
    ax.set_xlabel('RUVg factors')
    ax.set_title(a['label'], loc='left', fontsize=FS_ANNOT_SM)
    for s in ax.spines.values():
        s.set_visible(False)
    fig.savefig(FIG / f'{p}_signal_number_heatmap.png', dpi=300, bbox_inches='tight')
    print(t.to_string())


def upset(ax_bar, ax_dot, sets, top=UPSET_TOP, ylabels=True):
    names = list(sets)
    cols = [SET_COLOR[n] for n in names]
    memb = {}
    for i, n in enumerate(names):
        for g in sets[n]:
            memb[g] = memb.get(g, 0) | (1 << i)
    combos = [c for c in Counter(memb.values()).most_common() if c[0]][:top]
    x = np.arange(len(combos))
    bar_col = [cols[int(np.log2(m))] if m and not m & (m - 1) else '#555555' for m, _ in combos]
    ax_bar.bar(x, [c[1] for c in combos], color=bar_col, width=0.65)
    for xi, (_, v) in zip(x, combos):
        ax_bar.text(xi, v, f'{v:,}', ha='center', va='bottom', rotation=90)
    ax_bar.set_xlim(-0.6, len(combos) - 0.4)
    ax_bar.set_xticks([])
    ax_bar.set_ylim(0, max(v for _, v in combos) * 1.35)
    ax_bar.set_axisbelow(True)
    ax_bar.grid(axis='y', color='#DDDDDD', lw=0.8)
    for yi in range(len(names)):
        ax_dot.axhspan(yi - 0.5, yi + 0.5, color=cols[yi], alpha=0.10, lw=0)
    for xi, (mask, _) in zip(x, combos):
        on = [i for i in range(len(names)) if mask >> i & 1]
        ax_dot.plot([xi] * len(names), range(len(names)), 'o', ms=6, mfc='white', mec='#CCCCCC', mew=0.8)
        if len(on) > 1:
            ax_dot.plot([xi, xi], [min(on), max(on)], '-', lw=1.2, color='#555555', zorder=1)
        for i in on:
            ax_dot.plot([xi], [i], 'o', ms=6, mfc=cols[i], mec=cols[i], zorder=2)
    for i, n in enumerate(names):
        ax_dot.text(1.01, i, f'{len(sets[n]):,}', transform=ax_dot.get_yaxis_transform(),
                    ha='left', va='center', color=cols[i], fontsize=8)
    ax_dot.set_xlim(-0.6, len(combos) - 0.4)
    ax_dot.set_ylim(-0.6, len(names) - 0.4)
    ax_dot.set_xticks([])
    ax_dot.tick_params(axis='y', length=0)
    if ylabels:
        ax_dot.set_yticks(range(len(names)), names)
        for t, c in zip(ax_dot.get_yticklabels(), cols):
            t.set_color(c)
    else:
        ax_dot.set_yticks([])
    for sp in ax_dot.spines.values():
        sp.set_visible(False)


def null_inset(ax_bar, nmu_n, ns, draws):
    ax = ax_bar.inset_axes([0.62, 0.62, 0.46, 0.32])
    ax.barh([1, 0], [nmu_n, draws.mean()], 0.62, xerr=[0, draws.std()],
            color=['#B2182B', '#555555'], error_kw=dict(lw=0.8, capsize=2))
    for y, v, c in [(1, nmu_n, '#B2182B'), (0, draws.mean(), '#555555')]:
        ax.text(v + nmu_n * 0.03, y, f'{v:,.0f}', va='center', fontsize=7, color=c)
    ax.set_yticks([1, 0], ['patients', f'null (HC, n={ns})'], fontsize=7)
    ax.set_xlim(0, nmu_n * 1.6)
    ax.set_xticks([])
    ax.tick_params(axis='y', length=0)
    ax.set_title(f'union size, {nmu_n / max(draws.mean(), 1):.0f}x null', fontsize=7.5, loc='left', pad=2)
    for sp in ax.spines.values():
        sp.set_visible(False)


def upset_summary(a, p):
    fig = plt.figure(figsize=(5.5 * len(VGROUPS), 4.6))
    gs = fig.add_gridspec(2, len(VGROUPS), height_ratios=[2.0, 1.1], hspace=0.05, wspace=0.26)
    rows = []
    for k, grp in enumerate(VGROUPS):
        m = arm_mask(a, grp)
        sets = {DESIGN_NAME[d]: a['deg'][(CASE_OF[grp], d)] for d in DESIGN_ORDER}
        sets['Normative Modeling (Union)'] = nmu = set(genes[a['SIG'][m].any(axis=0)])
        ns, draws = arm_hc_draws(a, int(m.sum()))
        ax_bar = fig.add_subplot(gs[0, k])
        upset(ax_bar, fig.add_subplot(gs[1, k], sharex=ax_bar), sets, ylabels=(k == 0))
        ax_bar.set_title(f'{grp}  (n={int(m.sum())})', loc='left')
        if k == 0:
            ax_bar.set_ylabel('genes')
        null_inset(ax_bar, len(nmu), ns, draws)
        f = plt.figure(figsize=(7.5, 4.6))
        g = f.add_gridspec(2, 1, height_ratios=[2.0, 1.1], hspace=0.05)
        ab = f.add_subplot(g[0, 0])
        upset(ab, f.add_subplot(g[1, 0], sharex=ab), sets)
        ab.set_ylabel('genes')
        ab.set_title(f'{grp}  (n={int(m.sum())})', loc='left')
        null_inset(ab, len(nmu), ns, draws)
        f.savefig(FIG / f"{p}_upset_{grp.replace(' ', '_')}.png", dpi=300, bbox_inches='tight')
        plt.close(f)
        rows.append(dict(group=grp, n_pat=int(m.sum()), nm_union=len(nmu),
                         group_deg_k2=len(a['deg'][(CASE_OF[grp], RUVG_PICK)]),
                         shared_k2=len(a['deg'][(CASE_OF[grp], RUVG_PICK)] & nmu),
                         hc_draw_mean=round(draws.mean(), 1), hc_draw_sd=round(draws.std(), 1)))
    fig.savefig(FIG / f'{p}_upset_summary.png', dpi=300, bbox_inches='tight')
    summary = pd.DataFrame(rows)
    summary.to_csv(OUT / f'{p}_upset_group_vs_individual.csv', index=False)
    print(summary.to_string(index=False))

In [ ]:
# Panel figures, one call per (arm, panel): cutoff sweep vs DESeq2, marker recovery (hit grid + bars),
# the full q-tier heatmap, and the individual-vs-group Venn. Tissue panels use the tissue-marker hit rule.
GRID_BLOCKS = 1
CELL_W, CELL_H = 0.26, 0.14  # inches per marker column / patient row
LABEL_IN = 1.5               # inches reserved under the heatmap for the rotated gene names
LABELS['grid_full_x'] = 'every marker called at q < 0.20: {n_markers} genes in {n_pat_hit} of {n_pat} patients'
LABELS['grid_full_y'] = '{n_pat_hit} patients'
VENN_TOP = 14
LINE_H = 0.20          # data units per text line at FS_ANNOT_SM, linespacing included
C_GRP = '#7F7F7F'
PANEL_SPECS = {
    'pancreas': dict(tbl=marker_tbl, ylab='Pancreas marker genes called changed', title='pancreas cell-type markers (HPA)',
                     color='#B2182B', extra=['set'], detected_only=False, tissue=True,
                     row_colors=marker_tbl['set'].map(SET_C).values),
    'liver': dict(tbl=liver_tbl, ylab='Liver marker genes called changed', title='liver cell-type markers (HPA)',
                  color=LIVER_C, extra=['set'], detected_only=False, tissue=True, row_colors=None),
    'ot': dict(tbl=ot_tbl, ylab='OT pancreatic-cancer genes called changed', title='pancreatic-cancer genes (Open Targets)',
               color='#2166AC', extra=['score'], detected_only=True, tissue=False, row_colors=None),
}


def arm_sig(a, key, q, hc=False, null=False):
    tissue = PANEL_SPECS[key]['tissue']
    if null:
        Z = a['Z_null']
        return marker_sig_matrix(Z, q=q) if tissue else bh_sig_matrix(Z, q=q)
    Z = a['Z_hc'] if hc else a['Z']
    if tissue:
        return marker_sig_matrix(Z, q=q, names=None if hc else a['names'])
    return bh_sig_matrix(Z, q=q)


def shades_of(color):
    return [mcolors.to_hex(1 - w + w * np.array(mcolors.to_rgb(color))) for w in (1.0, 0.66, 0.38)]


def marker_sweep(a, p, key):
    sp = PANEL_SPECS[key]
    mtbl, color = sp['tbl'], sp['color']
    cols = np.array([gpos[e] for e in mtbl['ens']])
    use = mtbl['detected'].values if sp['detected_only'] else np.ones(len(mtbl), bool)
    sig_q = {q: arm_sig(a, key, q)[:, cols] for q in Q_GRID}
    sig_hc_q = {q: arm_sig(a, key, q, null=True)[:, cols] for q in Q_GRID}
    n_pat, n_hcm = len(a['names']), len(a['Z_null'])
    ns = min(n_pat, n_hcm)
    rng_s = np.random.default_rng(SEED)
    hc_draws = {q: np.array([int(sig_hc_q[q][rng_s.choice(n_hcm, ns, replace=False)][:, use].any(axis=0).sum())
                             for _ in range(N_SWEEP_DRAWS)]) for q in Q_GRID}
    hc_union = {q: int(np.median(hc_draws[q])) for q in Q_GRID}
    ens = mtbl['ens'].values[use]
    deseq = {d: [int((a['padj'][('PancreaticNeoplasm', d)]['padj'].reindex(ens).fillna(1.0).values < q).sum())
                 for q in Q_GRID] for d in DESIGN_ORDER}
    nm = [int((sig_q[q][:, use].sum(axis=0) > 0).sum()) for q in Q_GRID]

    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    for d in DESIGN_ORDER:
        ax.plot(Q_GRID, deseq[d], 'o-', color=SET_COLOR[DESIGN_NAME[d]], label=DESIGN_NAME[d], markersize=8, lw=1.5)
    ax.plot(Q_GRID, nm, 'o-', color=color, lw=2.5, label='Normative Modeling (Union)', markersize=8)
    ax.plot(Q_GRID, [hc_union[q] for q in Q_GRID], 'o--', color='#4D4D4D', lw=1.5, markersize=8,
            label=f'Normative Null (HC, Union, n={ns} of {n_hcm})')
    ax.fill_between(Q_GRID, [int(hc_draws[q].min()) for q in Q_GRID], [int(hc_draws[q].max()) for q in Q_GRID],
                    color='#4D4D4D', alpha=0.18, lw=0)
    ax.set_xscale('log')
    ax.set_xticks(Q_GRID, [str(q) for q in Q_GRID])
    ax.set_xlabel(LABELS['sweep_x'])
    ax.set_ylabel(LABELS['sweep_y'].format(ylab=sp['ylab'], n_panel=len(ens)))
    ax.set_title(a['label'], loc='left', fontsize=FS_ANNOT_SM)
    ax.grid(zorder=0, alpha=0.3)
    ax.legend(frameon=False, loc='upper left')
    fig.savefig(FIG / f'{p}_{key}_sweep.png', dpi=300, bbox_inches='tight')
    out = pd.DataFrame({'cutoff': Q_GRID, 'markers': nm,
                        'patients': [int((sig_q[q][:, use].sum(axis=1) > 0).sum()) for q in Q_GRID],
                        'hc_markers': [hc_union[q] for q in Q_GRID],
                        'hc_markers_max': [int(hc_draws[q].max()) for q in Q_GRID],
                        'excess_over_hc_max': [n - int(hc_draws[q].max()) for n, q in zip(nm, Q_GRID)],
                        'p_emp': [float((hc_draws[q] >= n).mean()) for n, q in zip(nm, Q_GRID)],
                        'n_pat': n_pat, 'n_hc_pool': n_hcm, 'n_hc_drawn': ns, 'n_draws': N_SWEEP_DRAWS, **deseq})
    out.to_csv(OUT / f'{p}_{key}_sweep.csv', index=False)
    print(out.to_string(index=False))


def marker_recovery(a, p, key):
    sp = PANEL_SPECS[key]
    mtbl, color, row_colors = sp['tbl'], sp['color'], sp['row_colors']
    stage = a['stage']
    cols = np.array([gpos[e] for e in mtbl['ens']])
    sig_A = {q: arm_sig(a, key, q)[:, cols] for q in Q_TIERS}
    use = mtbl['detected'].values if sp['detected_only'] else np.ones(len(mtbl), bool)
    tier = np.zeros(sig_A[Q_TIERS[0]].shape)
    for i, q in enumerate(sorted(Q_TIERS, reverse=True)):
        tier[sig_A[q]] = i + 1
    tier[:, ~use] = 0
    shades = shades_of(color)

    fig_grid, ax = plt.subplots(figsize=(11.0, 7.0))
    hit = tier > 0
    cand = list(np.where(hit.any(axis=0))[0])
    rows, shown = [], np.zeros(tier.shape[0], bool)
    while cand and len(rows) < TOP_ROWS:
        gain = [(int((hit[:, c] & ~shown).sum()), int(tier[:, c].max()), int(hit[:, c].sum())) for c in cand]
        if max(g[0] for g in gain) == 0:
            shown[:] = False
            continue
        best = cand[max(range(len(cand)), key=lambda i: gain[i])]
        rows.append(best)
        shown |= hit[:, best]
        cand.remove(best)
    rows = np.array(rows, dtype=int)
    rows = rows[np.lexsort((mtbl['symbol'].values[rows], -tier[:, rows].max(axis=0),
                            -(tier[:, rows] == len(Q_TIERS)).sum(axis=0)))]
    keep = np.where(tier.max(axis=1) > 0)[0]
    keep = keep[np.lexsort((-(tier[np.ix_(keep, rows)] == len(Q_TIERS)).sum(axis=1),
                            [list(STAGE_C).index(s) for s in stage[keep]]))]
    T = tier[np.ix_(keep, rows)].T
    ax.imshow(np.zeros(T.shape), cmap='Greys', vmin=0, vmax=1, aspect='auto')
    tier_h = []
    for lvl, sh in zip(range(len(Q_TIERS), 0, -1), shades):
        for y, x in zip(*np.where(T == lvl)):
            ax.add_patch(plt.Rectangle((x - 0.43, y - 0.43), 0.86, 0.86, color=sh, lw=0, zorder=2))
        tier_h.append(plt.Rectangle((0, 0), 1, 1, color=sh))
    for j, sm in enumerate(stage[keep]):
        ax.add_patch(plt.Rectangle((j - 0.5, -0.5), 1, len(rows), color=STAGE_C[sm], alpha=0.12, lw=0, zorder=0))
    ax.set_xlim(-0.5, len(keep) - 0.5)
    ax.set_ylim(len(rows) - 0.5, -0.5)
    ax.set_xticks(np.arange(-0.5, len(keep), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(rows), 1), minor=True)
    ax.grid(which='minor', color='#FFFFFF', lw=0.8)
    ax.set_yticks(range(len(rows)), [mtbl['symbol'].values[r] for r in rows])
    if row_colors is not None:
        for t, r in zip(ax.get_yticklabels(), rows):
            t.set_color(row_colors[r])
    ax.set_xticks([])
    ax.set_xlabel(LABELS['grid_x'].format(n_pat_hit=len(keep), n_pat=tier.shape[0]))
    ax.set_ylabel(LABELS['grid_y'].format(n_rows=len(rows), n_markers_hit=int((tier.max(axis=0) > 0).sum())))
    ax.set_title(a['label'], loc='left', fontsize=FS_ANNOT_SM)
    ax.tick_params(length=0)
    h2 = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.35) for c in STAGE_C.values()]
    l1 = [f'q < {q:.2f}' for q in sorted(Q_TIERS)]
    ax.legend([x for pair in zip(tier_h, h2) for x in pair], [x for pair in zip(l1, STAGE_C) for x in pair],
              frameon=False, loc='upper left', bbox_to_anchor=(0.1, -0.06), ncol=3,
              handletextpad=0.3, columnspacing=1.2, labelspacing=0.6)
    for s in ax.spines.values():
        s.set_visible(False)
    fig_grid.savefig(FIG / f'{p}_{key}_recovery_grid.png', dpi=300, bbox_inches='tight')

    fig_bar, ax = plt.subplots(figsize=(3, 6.4))
    pat_f = [(sig_A[q][:, use].sum(axis=1) > 0).mean() for q in sorted(Q_TIERS)]
    mrk_f = [(sig_A[q][:, use].sum(axis=0) > 0).mean() for q in sorted(Q_TIERS)]
    xs = [0, BAR_GAP]
    for x, (fr, n) in zip(xs, zip([pat_f, mrk_f], [tier.shape[0], int(use.sum())])):
        bot = 0
        for seg, sh in zip([fr[0], fr[1] - fr[0], fr[2] - fr[1]], shades):
            ax.bar(x, seg, bottom=bot, width=BAR_W, color=sh, lw=0)
            bot += seg
        ax.bar(x, 1 - bot, bottom=bot, width=BAR_W, color='#EFEFEF', lw=0)
        ax.text(x, bot + 0.015, f'{bot:.0%}', ha='center', va='bottom', fontsize=12, fontweight='bold')
        ax.text(x, 1.04, LABELS['bar_n'].format(n=n), ha='center', va='bottom', fontsize=12)
    ax.set_xlim(-0.5 - BAR_W / 2, BAR_GAP + 0.5 + BAR_W / 2)
    ax.set_xticks(xs, list(LABELS['bar_x']))
    ax.set_ylim(0, 1.14)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ['0', '25%', '50%', '75%', '100%'])
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    fig_bar.savefig(FIG / f'{p}_{key}_recovery_bars.png', dpi=300, bbox_inches='tight')
    out = pd.DataFrame({'cutoff': sorted(Q_TIERS),
                        'markers': [int((sig_A[q][:, use].sum(axis=0) > 0).sum()) for q in sorted(Q_TIERS)],
                        'patients': [int((sig_A[q][:, use].sum(axis=1) > 0).sum()) for q in sorted(Q_TIERS)],
                        'n_patients': tier.shape[0], 'n_panel': int(use.sum())})
    out.to_csv(OUT / f'{p}_{key}_recovery.csv', index=False)
    print(out.to_string(index=False))


def marker_heatmap(a, p, key):
    sp = PANEL_SPECS[key]
    mtbl, color, stage = sp['tbl'], sp['color'], a['stage']
    cols = np.array([gpos[e] for e in mtbl['ens']])
    use = mtbl['detected'].values
    sig_q = {q: arm_sig(a, key, q)[:, cols] & use[None, :] for q in Q_TIERS}
    tier = np.zeros(sig_q[Q_TIERS[0]].shape)
    for i, q in enumerate(sorted(Q_TIERS, reverse=True)):
        tier[sig_q[q]] = i + 1
    mk = np.where(tier.max(axis=0) > 0)[0]
    mk = mk[np.lexsort((mtbl['symbol'].values[mk], -(tier[:, mk] > 0).sum(axis=0), -tier[:, mk].max(axis=0)))]
    pt = np.where(tier[:, mk].max(axis=1) > 0)[0]
    pt = pt[np.lexsort((-(tier[np.ix_(pt, mk)] == len(Q_TIERS)).sum(axis=1),
                        [list(STAGE_C).index(s) for s in stage[pt]]))]
    shades = shades_of(color)
    grid_in = CELL_H * len(pt)
    fig, ax = plt.subplots(figsize=(CELL_W * len(mk) + 1.0, grid_in + LABEL_IN))
    T = tier[np.ix_(pt, mk)]
    ax.imshow(np.zeros(T.shape), cmap='Greys', vmin=0, vmax=1, aspect='auto')
    for lvl, sh in zip(range(len(Q_TIERS), 0, -1), shades):
        for y, x in zip(*np.where(T == lvl)):
            ax.add_patch(plt.Rectangle((x - 0.43, y - 0.43), 0.86, 0.86, color=sh, lw=0, zorder=2))
    for i, sm in enumerate(stage[pt]):
        ax.add_patch(plt.Rectangle((-0.5, i - 0.5), len(mk), 1, color=STAGE_C[sm], alpha=0.12, lw=0, zorder=0))
    ax.set_xlim(-0.5, len(mk) - 0.5)
    ax.set_ylim(len(pt) - 0.5, -0.5)
    ax.set_xticks(np.arange(-0.5, len(mk), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(pt), 1), minor=True)
    ax.grid(which='minor', color='#FFFFFF', lw=0.8)
    ax.set_xticks(range(len(mk)), mtbl['symbol'].values[mk], rotation=90)
    ax.set_yticks([])
    ax.set_ylabel(LABELS['grid_full_y'].format(n_pat_hit=len(pt)))
    ax.set_title(a['label'], loc='left', fontsize=FS_ANNOT_SM)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    h1 = [plt.Rectangle((0, 0), 1, 1, color=sh) for sh in shades]
    h2 = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.35) for c in STAGE_C.values()]
    ax.legend([x for pair in zip(h1, h2) for x in pair],
              [x for pair in zip([f'q < {q:.2f}' for q in sorted(Q_TIERS)], STAGE_C) for x in pair],
              frameon=False, loc='upper left', bbox_to_anchor=(0.35, -LABEL_IN / grid_in), ncol=3,
              handletextpad=0.3, columnspacing=1.2)
    ax.set_xlabel(LABELS['grid_full_x'].format(n_markers=len(mk), n_pat_hit=len(pt), n_pat=len(stage)))
    fig.savefig(FIG / f'{p}_{key}_heatmap.png', dpi=300, bbox_inches='tight')
    mtbl.iloc[mk].assign(**{f'n_pat_q{q:g}': sig_q[q][:, mk].sum(axis=0) for q in Q_TIERS}).to_csv(
        OUT / f'{p}_{key}_heatmap.csv', index=False)
    print(f'{p} {key} heatmap: {len(mk)} markers x {len(pt)} patients; '
          f'markers by cutoff { {q: int(sig_q[q].any(axis=0).sum()) for q in sorted(Q_TIERS)} }')


def case_table(a, key):
    sp = PANEL_SPECS[key]
    t = sp['tbl'].copy()
    c = np.array([gpos[e] for e in t['ens']])
    t['n_pat_bh'] = arm_sig(a, key, FDR_Q)[:, c].sum(axis=0)
    t['n_hc_bh'] = arm_sig(a, key, FDR_Q, hc=True)[:, c].sum(axis=0)
    t['group_padj'] = t['ens'].map(a['best_padj'])
    t['group_lfc'] = t['ens'].map(a['best_lfc'])
    t['indiv'] = t['n_pat_bh'] > 0
    t['group'] = t['group_padj'].fillna(1.0) < DESEQ2_PADJ
    t['measured'] = t['detect_frac'] > 0
    return t.sort_values(['n_pat_bh', 'group_padj'], ascending=[False, True])


def venn(a, p, key):
    sp = PANEL_SPECS[key]
    t = case_table(a, key)
    t.to_csv(OUT / f'{p}_{key}_case_table.csv', index=False)
    ok = t['measured']
    both, io = t[ok & t['indiv'] & t['group']], t[ok & t['indiv'] & ~t['group']]
    go = t[ok & ~t['indiv'] & t['group']].sort_values('group_padj')

    def block(df):
        s = df['symbol'].head(VENN_TOP).tolist()
        body = (s if s else ['(none)']) + ([f'+{len(df) - len(s)} more'] if len(df) > len(s) else [])
        return f'[{len(df)}]\n' + '\n'.join(body), len(body) + 1

    with plt.rc_context({k: FS_ANNOT_SM for k in ('font.size', 'axes.titlesize')}):
        fig, ax = plt.subplots(figsize=(6.5, 5.6))
        blocks = [block(d) for d in (io, both, go)]
        h = max(n for _, n in blocks) * LINE_H
        for x, c, lab, n in ((-1, sp['color'], 'Individual\nBH q<0.05, >=1 patient', len(both) + len(io)),
                             (1, C_GRP, 'Group\nDESeq2 padj<0.05 (any run)', len(both) + len(go))):
            ax.add_patch(Ellipse((x, 0), 4.0, h + 1.0, facecolor=c, alpha=0.15, edgecolor=c, lw=1.2))
            ax.text(x * 1.9, h / 2 + 0.6, f'{lab}\nn={n}', ha='center', va='bottom', color=c)
        for x, (txt, _) in zip((-1.55, 0.0, 1.55), blocks):
            ax.text(x, h / 2, txt, ha='center', va='top', linespacing=1.4)
        ax.text(0, -h / 2 - 0.95, f'neither: {int((ok & ~t["indiv"] & ~t["group"]).sum())} genes silent on both sides',
                ha='center', va='top', color='0.35', fontsize=10)
        ax.set_title(f"{sp['title']} -- {a['label']}", fontsize=FS_ANNOT_SM + 2)
        ax.set_xlim(-3.3, 3.3)
        ax.set_ylim(-h / 2 - 1.7, h / 2 + 1.6)
        ax.set_aspect('equal')
        ax.axis('off')
        fig.savefig(FIG / f'{p}_{key}_venn.png', dpi=300, bbox_inches='tight')
    print(f'{p} {key}: individual {len(both) + len(io)}, group {len(both) + len(go)}, both {len(both)}; '
          f'also BH-significant in >=1 HC: {int((ok & (t["n_hc_bh"] > 0)).sum())}')


def panel_figures(p, key):
    a = ARMS[p]
    marker_sweep(a, p, key)
    marker_recovery(a, p, key)
    marker_heatmap(a, p, key)
    venn(a, p, key)

# Part A -- Moore Batch_1 + Batch_2 + Batch_3 (pooled DESeq2, our own runs)

72 neoplastic patients and 69 HC, exactly the samples of `pancreatic_deg_pooled`. The DESeq2 side is our own
pooled-cohort run with RUVg k=1-3, not a reproduction of Moore et al.

Signal Number Heatmap -> UpSet, then pancreas -> liver -> Open Targets, each as sweep -> marker recovery -> heatmap -> Venn.

In [ ]:
signal_number_heatmap(ARMS['A'], 'A')
upset_summary(ARMS['A'], 'A')

### A. Pancreas cell-type markers

In [ ]:
panel_figures('A', 'pancreas')

### A. Liver cell-type markers

In [ ]:
panel_figures('A', 'liver')

### A. Open Targets pancreatic-cancer genes

In [ ]:
panel_figures('A', 'ot')

# Part B -- Moore Batch_1 only (matched to DESeq2)

39 neoplastic patients and 48 HC, exactly the samples of `pancreatic_deg` (the CEDAR cohort Moore et al. fitted on).

Signal Number Heatmap -> UpSet, then pancreas -> liver -> Open Targets, each as sweep -> marker recovery -> heatmap -> Venn.

In [ ]:
signal_number_heatmap(ARMS['B'], 'B')
upset_summary(ARMS['B'], 'B')

### B. Pancreas cell-type markers

In [ ]:
panel_figures('B', 'pancreas')

### B. Liver cell-type markers

In [ ]:
panel_figures('B', 'liver')

### B. Open Targets pancreatic-cancer genes

In [ ]:
panel_figures('B', 'ot')

# Part C -- supporting analyses

Severity gradient, per-sample marker hits and tissue table (all arm); matched-arm tables (Batch_1: group vs individual, representativeness, Jaccard, DESeq2 on the markers, marker ledger), the DESeq2-only genes, and the comparison with Moore et al.

In [ ]:
# Severity gradient: per-sample deviation burden across the neoplastic series, with acute
# pancreatitis (non-neoplastic, same three batches) as the inflammatory contrast. All arm --
# no group contrast is involved, so every Moore Batch_1-3 sample is used.
from scipy.stats import kruskal, spearmanr

SEV_ORDER = NEOPLASTIC + ['Pancreatitis']
Z_panc = src['Z_dis'][a_panc]
sev = pd.concat([per_sample[['stage', 'n_sig']],
                 pd.DataFrame(dict(stage='Pancreatitis', n_sig=bh_sig_matrix(Z_panc).sum(axis=1)))])

neo_sev = sev[sev['stage'].isin(NEOPLASTIC)]
rho, p_rho = spearmanr(neo_sev['stage'].map({k: i for i, k in enumerate(NEOPLASTIC)}), neo_sev['n_sig'])
kw = kruskal(*[d['n_sig'].values for _, d in neo_sev.groupby('stage')]).pvalue
print(sev.groupby('stage')['n_sig'].agg(n='size', median='median', mean='mean')
      .reindex(SEV_ORDER).round(1).to_string())
print(f'neoplastic gradient: Spearman rho={rho:+.3f} p={p_rho:.2e} (n={len(neo_sev)}), '
      f'Kruskal p={kw:.2e}')

In [ ]:
SEV_C = {**STAGE_C, 'Pancreatitis': '#999999'}
SEV_LAB = ['IPMN', 'Islet cell\ntumour', 'PDAC', 'Acute\npancreatitis']
FS = {k: FS_ANNOT_SM for k in ('font.size', 'axes.labelsize', 'xtick.labelsize',
                               'ytick.labelsize', 'legend.fontsize')}
vals = [sev.loc[sev['stage'] == s, 'n_sig'].values for s in SEV_ORDER]
med = [np.median(v) for v in vals[:len(NEOPLASTIC)]]
q1, q3 = np.percentile(sev['n_sig'], [25, 90])

with plt.rc_context(FS):
    fig, ax = plt.subplots(figsize=(2.3, 3.8))
    bp = ax.boxplot(vals, widths=0.5, patch_artist=True,
                    flierprops=dict(marker='o', ms=1.5, mfc='0.45', mec='none'),
                    medianprops=dict(color='black', lw=1.4))
    for box, s in zip(bp['boxes'], SEV_ORDER):
        box.set(facecolor=SEV_C[s], alpha=0.75, lw=0.9)
    ax.plot(range(1, len(NEOPLASTIC) + 1), med, color='#D90429', ls='--', marker='o', lw=1.5, ms=4,
            zorder=5)
    for i, v in enumerate(med):
        ax.annotate(f'{v:.0f}', (i + 1, v), textcoords='offset points', xytext=(0, 7), ha='center',
                    color='#D90429')
    ax.axvline(len(NEOPLASTIC) + 0.5, color='0.6', ls=':', lw=1.0)
    ax.text(0.03, 0.99, f'Spearman rho = {rho:+.2f}, p = {p_rho:.1e}\nKruskal p = {kw:.1e}',
            transform=ax.transAxes, va='top')
    ax.set_xticks(range(1, len(SEV_ORDER) + 1), SEV_LAB)
    ax.set_ylabel('BH-significant genes per sample ($q=0.05$)')
    ax.set_ylim(-2, (q3 + 1.5 * (q3 - q1)) * 1.2)
    fig.savefig(FIG / 'fig_severity_gradient.png', dpi=300, bbox_inches='tight')

In [ ]:
DET = marker_tbl['detected'].values
# hit level, not a cutoff sweep: no group contrast is on the other side, so this runs on the
# all arm (Moore Batch_1-3) rather than the DESeq2-matched subset
hit_pat, hit_hc = SIG_A_M[:, mcols], SIG_HC_A_M[:, mcols]


def set_count(c, thr):
    a = int((hit_pat[:, c].sum(axis=1) >= thr).sum())
    b = int((hit_hc[:, c].sum(axis=1) >= thr).sum())
    return dict(n_genes=int(c.sum()), min_hits=thr, pat=f'{a}/{len(hit_pat)}', hc=f'{b}/{len(hit_hc)}',
                pat_frac=a / len(hit_pat), hc_frac=b / len(hit_hc))


rows = []
for s in list(MARKER_SETS) + ['all']:
    base = np.ones(len(DET), bool) if s == 'all' else (marker_tbl['set'].values == s)
    for panel, flag in (('full', base), ('detected', base & DET)):
        if flag.sum() == 0:
            continue
        for thr in (1, 2):
            rows.append(dict(set=s, panel=panel, **set_count(flag, thr)))
indiv = pd.DataFrame(rows)
indiv['headline'] = (indiv.panel == 'detected') & (indiv.min_hits == 1) & (indiv['set'] != 'all')
indiv.to_csv(OUT / 'marker_individual_hits.csv', index=False)
print(f'per-sample deviation on the markers (BH up, patient reads >= {MIN_HIT_READS}), counts only -- no patient-group vs')
print('HC-group test is computed here: pooling samples into two groups and testing the means is the')
print('operation this notebook is arguing against. The HC column is the reference to read by eye.')
print(indiv.round(4).to_string(index=False))

prof = pd.DataFrame(dict(sample=names_A, stage=stage_A, n_hit=hit_pat[:, DET].sum(axis=1)))
for s in MARKER_SETS:
    prof[f'n_{s}'] = hit_pat[:, (marker_tbl['set'].values == s) & DET].sum(axis=1)
sym_det = marker_tbl['symbol'].values
prof['markers'] = [' '.join(f'{sym_det[j]}{Zm_A[i, j]:+.1f}'
                            for j in np.where(hit_pat[i] & DET)[0]) for i in range(len(prof))]
prof['markers_undetected'] = [' '.join(f'{sym_det[j]}{Zm_A[i, j]:+.1f}'
                                       for j in np.where(hit_pat[i] & ~DET)[0]) for i in range(len(prof))]
prof = prof.sort_values('n_hit', ascending=False)
prof.to_csv(OUT / 'marker_per_patient_profile.csv', index=False)
print('\nper-patient profiles on detected markers only (top 10); the last column is what the '
      'zero/low-count columns would have added as noise')
print(prof.head(10)[['sample', 'stage', 'n_hit', 'markers']].to_string(index=False))
print('\nby stage, frac with >=1 hit on detected markers:')
print(pd.DataFrame({'stage': stage_A, 'h': hit_pat[:, DET].sum(axis=1) > 0})
      .groupby('stage', observed=True)['h'].agg(n='size', frac='mean').round(2).to_string())

In [ ]:
# Which sample carries which tissue marker (all arm, tissue-marker hit rule)
with h5py.File(config.H5AD_PATH, 'r') as h:
    obs_i = dec(h['obs/_index'][()])
    meta = {}
    for c in META_COLS:
        o = h['obs'][c]
        meta[c] = (pd.Categorical.from_codes(o['codes'][()], dec(o['categories'][()])).astype(str)
                   if isinstance(o, h5py.Group) else np.asarray(o[()]))
    meta = pd.DataFrame(meta, index=obs_i)

tcols = np.array([gpos[e] for e in tissue_tbl['ens']])
SIG_T = np.vstack([SIG_A_M[:, tcols], SIG_HC_A_M[:, tcols]])
Z_T = np.vstack([Z_A[:, tcols], Z_hc_A[:, tcols]])
DET_T = tissue_tbl['detected'].values
syms_T, sets_T = tissue_tbl['symbol'].values, tissue_tbl['set'].values
names_T = np.concatenate([names_A, names_hc_A])

tbl = pd.DataFrame(dict(sample=names_T,
                        group=np.where(np.arange(len(names_T)) < len(names_A), 'patient', 'HC'),
                        stage=np.concatenate([stage_A, np.repeat('Healthy Control', len(names_hc_A))]),
                        n_sig=np.concatenate([SIG_A.sum(axis=1), SIG_HC_A.sum(axis=1)])))
for c in META_COLS:
    tbl[c] = tbl['sample'].map(meta[c])
for tis in TISSUES:
    c = (sets_T == tis) & DET_T
    tbl[f'n_{tis}'] = SIG_T[:, c].sum(axis=1)
    tbl[tis] = [' '.join(f'{s}{z:+.1f}' for s, z in
                         zip(syms_T[c][SIG_T[i, c]], Z_T[i, c][SIG_T[i, c]]))
                for i in range(len(tbl))]
tbl['n_tissue'] = tbl[[f'n_{t}' for t in TISSUES]].sum(axis=1)
tbl = tbl.sort_values(['group', 'n_tissue'], ascending=[True, False])
tbl.to_csv(OUT / 'tissue_marker_per_sample.csv', index=False)

SHOW = ['sample', 'stage', 'Tumor_Stage', 'Met_Site', 'n_sig'] + [f'n_{t}' for t in TISSUES]
print(f'\nper-sample tissue marker hits (BH q<0.05, detected panel) -> tissue_marker_per_sample.csv')
print('patients carrying at least one, top 25:')
print(tbl[(tbl.group == 'patient') & (tbl.n_tissue > 0)].head(25)[SHOW].to_string(index=False))

print('\nby metastasis site (patients) -- the check this notebook has to survive, not pass:')
pat = tbl[tbl.group == 'patient'].copy()
pat['met'] = pat['Met_Site'].str.lower().map(
    lambda v: 'liver' if v == 'liver' else 'lymph node' if 'lymph' in str(v) else
    'none' if v in ('none', 'no') else 'not recorded')
summ = pat.groupby('met').agg(n=('sample', 'size'), median_n_sig=('n_sig', 'median'),
                              **{f'pct_{t}': (f'n_{t}', lambda s: f'{(s > 0).mean():.0%}')
                                 for t in TISSUES})
hc = tbl[tbl.group == 'HC']
summ.loc['HC (reference)'] = [len(hc), hc['n_sig'].median()] + \
    [f'{(hc[f"n_{t}"] > 0).mean():.0%}' for t in TISSUES]
print(summ.to_string())
print('\nliver metastasis is perfectly confounded with Tumor_Stage here:')
print(pd.crosstab(pat['Tumor_Stage'], pat['met']).to_string())

In [ ]:
rng = np.random.default_rng(SEED)
n_hc = SIG_HC_NULL.shape[0]   # the null pool, not every Batch_1 control


def group_mask(grp):
    return np.ones(len(names_neo), bool) if grp == 'all neoplasm' else (stage_neo == grp)

def hc_union_draws(n, cols=None, M=None):
    ns = min(n, n_hc)
    M = SIG_HC_NULL if M is None else M
    M = M if cols is None else M[:, cols]
    return ns, np.array([int(M[rng.choice(n_hc, ns, replace=False)].any(axis=0).sum())
                         for _ in range(N_HC_DRAWS)])

rows = []
for design in DESIGN_PAIR:
    for grp, case in CASE_OF.items():
        m = group_mask(grp)
        nmu = set(genes[SIG[m].any(axis=0)])
        D = deg[(case, design)]
        ns, draws = hc_union_draws(int(m.sum()))
        hcu = int(np.median(draws))
        rows.append(dict(design=design, group=grp, n_pat=int(m.sum()), n_hc_drawn=ns,
                         deg_n=len(D), nm_union=len(nmu), hc_union=hcu,
                         nm_over_hc=round(len(nmu) / max(hcu, 1), 2),
                         shared=len(nmu & D), novel=len(nmu - D),
                         deg_orphan=len(D - nmu),
                         deg_orphan_frac=round(len(D - nmu) / max(len(D), 1), 3)))
matched = pd.DataFrame(rows)
matched.to_csv(OUT / 'matched_group_vs_individual.csv', index=False)
print(matched.to_string(index=False))
print('\nn_pat / n_hc_drawn are SAMPLE counts; deg_n / nm_union / hc_union are GENE counts.')
print('the comparison is nm_union vs hc_union at n_pat vs n_hc_drawn samples.')

In [ ]:
carriers = SIG.sum(axis=0)
hc_carriers = SIG_HC.sum(axis=0)
pd.DataFrame(dict(gene=genes, n_patients_sig=carriers, n_hc_sig=hc_carriers)).to_csv(
    OUT / 'gene_carrier_table.csv', index=False)

leg1b = []
for design in DESIGN_PAIR:
    D = sorted(deg[('PancreaticNeoplasm', design)] | deg[('PDAC', design)])
    idx = np.array([gpos[g] for g in D])
    Zu = Z_neo[:, idx]
    mean_z, sd_z = Zu.mean(axis=0), Zu.std(axis=0, ddof=1)
    leg1b.append(pd.DataFrame(dict(design=design, gene=D, mean_z=mean_z, sd_z=sd_z,
                                   abs_mean_over_sd=np.abs(mean_z) / sd_z,
                                   sign_concordance=np.mean(np.sign(Zu) == np.sign(mean_z)[None, :], axis=0),
                                   n_patients_sig=carriers[idx])))
leg1b = pd.concat(leg1b, ignore_index=True)
leg1b.to_csv(OUT / 'mean_representativeness.csv', index=False)

print('representativeness of the group mean over its own DE genes (PDAC + pooled contrast)')
print(leg1b.groupby('design')[['abs_mean_over_sd', 'sign_concordance']]
      .describe(percentiles=[0.5]).round(3).to_string())
for d, sub in leg1b.groupby('design'):
    print(f"  {d}: |mean|/SD < 0.5 in {(sub.abs_mean_over_sd < 0.5).mean():.0%}, "
          f"sign concordance < 0.7 in {(sub.sign_concordance < 0.7).mean():.0%}")

In [ ]:
def pair_jaccard(M, max_pairs=4000, seed=SEED):
    r = np.random.default_rng(seed)
    n = M.shape[0]
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    if len(pairs) > max_pairs:
        pairs = [pairs[k] for k in r.choice(len(pairs), max_pairs, replace=False)]
    out = []
    for i, j in pairs:
        u = int((M[i] | M[j]).sum())
        if u:
            out.append(int((M[i] & M[j]).sum()) / u)
    return np.array(out)


def size_matched_null(M, seed=SEED):
    r = np.random.default_rng(seed)
    N = M.shape[1]
    Mn = np.zeros_like(M)
    for i, k in enumerate(M.sum(axis=1)):
        if k:
            Mn[i, r.choice(N, int(k), replace=False)] = True
    return Mn


j_pat = pair_jaccard(SIG)
j_hc = pair_jaccard(SIG_HC)
j_null = pair_jaccard(size_matched_null(SIG))
spec = pd.DataFrame([dict(set_='patients', n_pairs=len(j_pat), median=np.median(j_pat), mean=j_pat.mean()),
                     dict(set_='HC (batch-matched)', n_pairs=len(j_hc), median=np.median(j_hc), mean=j_hc.mean()),
                     dict(set_='size-matched null', n_pairs=len(j_null), median=np.median(j_null), mean=j_null.mean())])
spec.to_csv(OUT / 'patient_specificity_jaccard.csv', index=False)
print(spec.round(4).to_string(index=False))

share = pd.Series(carriers[carriers > 0]).value_counts().sort_index()
carrier_dist = share.rename('n_genes').rename_axis('n_patients_sharing').reset_index()
carrier_dist.to_csv(OUT / 'gene_carrier_distribution.csv', index=False)
print('\nhow many patients share a gene (the group-list premise is that this mass sits high)')
print(carrier_dist.head(10).to_string(index=False))
print(f"genes carried by exactly 1 patient: {int(share.get(1, 0))} of {int((carriers > 0).sum())} "
      f"({share.get(1, 0) / max((carriers > 0).sum(), 1):.0%})")

In [ ]:
grp_rows = []
for f in sorted(DEG_DIR.glob('moore_b1__*/*.csv.gz')):
    case, design = f.parent.name.split('__')[1], f.name.split('.')[0]
    if case not in ('PDAC', 'PancreaticNeoplasm'):
        continue
    r = pd.read_csv(f, index_col=0)
    for s in MARKER_SETS:
        ens = [e for e in marker_tbl.loc[marker_tbl['set'] == s, 'ens'] if e in r.index]
        sub = r.loc[ens]
        grp_rows.append(dict(case=case, design=design, set=s,
                             n_in_table=len(sub), n_tested=int(sub.padj.notna().sum()),
                             n_padj05=int((sub.padj < 0.05).sum()),
                             n_rawp05=int((sub.pvalue < 0.05).sum()),
                             min_padj=sub.padj.min(), med_lfc=sub.log2FoldChange.median(),
                             all_sig_transcriptome=int((r.padj < 0.05).sum())))
group_side = pd.DataFrame(grp_rows)
group_side.to_csv(OUT / 'marker_group_deseq2.csv', index=False)
print('what the group contrast sees on the same markers')
print(group_side.round(3).to_string(index=False))

nominal = []
for f in sorted(DEG_DIR.glob('moore_b1__*/*.csv.gz')):
    case, design = f.parent.name.split('__')[1], f.name.split('.')[0]
    if case not in ('PDAC', 'PancreaticNeoplasm'):
        continue
    r = pd.read_csv(f, index_col=0)
    ens = [e for e in marker_tbl['ens'] if e in r.index]
    sub = r.loc[ens]
    for e, row in sub[sub.pvalue < 0.05].iterrows():
        nominal.append(dict(case=case, design=design, symbol=ens_of[e][1], set=ens_of[e][0],
                            lfc=row.log2FoldChange, pvalue=row.pvalue, padj=row.padj))
print('\nmarkers reaching raw p<0.05 in any group design (none reach padj<0.05):')
print(pd.DataFrame(nominal).round(4).to_string(index=False) if nominal else '  none')

In [ ]:
# Act 3 headline: on the same marker panel, how many genes does each view recover? Group side =
# DESeq2 padj<0.05. Individual side = per-sample BH q<0.05 in >=1 patient (act 2's union restricted
# to the panel). Control = that union on batch-matched HC at the same n, median and max over draws.
PANELS = {'full': np.ones(len(marker_tbl), bool), 'detected': marker_tbl['detected'].values}

rows = []
for pname, pmask in PANELS.items():
    pens = marker_tbl['ens'].values[pmask]
    pcols = np.array([gpos[e] for e in pens])
    pset = set(pens)
    for design in DESIGN_PAIR:
        for grp, case in CASE_OF.items():
            m = group_mask(grp)
            nm_m = set(pens[SIG_M[m][:, pcols].any(axis=0)])
            ns, draws = hc_union_draws(int(m.sum()), cols=pcols, M=SIG_HC_NULL_M)
            rows.append(dict(panel=pname, n_panel=int(pmask.sum()), design=design, group=grp,
                             n_pat=int(m.sum()), n_hc_drawn=ns,
                             group_deg=len(pset & deg[(case, design)]), nm_union=len(nm_m),
                             hc_union_med=float(np.median(draws)), hc_union_max=int(draws.max()),
                             nm_minus_hcmax=len(nm_m) - int(draws.max())))
recovery = pd.DataFrame(rows)
recovery.to_csv(OUT / 'marker_recovery_group_vs_union.csv', index=False)
print('marker GENES recovered by each view. n_pat / n_hc_drawn are SAMPLE counts -- they say the')
print('HC baseline was built at the same size, not how many genes it found. Read group_deg vs')
print('nm_union vs hc_union_med, all three of them gene counts on the same panel.')
print(recovery.to_string(index=False))

any_group = set().union(*deg.values())
for pname, pmask in PANELS.items():
    pens = set(marker_tbl['ens'].values[pmask])
    print(f'{pname} panel (n={len(pens)}): markers at padj<0.05 in ANY of the {len(deg)} '
          f'group runs = {len(pens & any_group)}')

In [ ]:
# The named genes behind those counts, with the best result the group side achieves anywhere
# across all 20 runs placed next to the per-sample result.
allres = pd.concat(padj_of.values())
best_padj = allres.groupby(level=0)['padj'].min()
best_rawp = allres.groupby(level=0)['pvalue'].min()
best_lfc = allres.groupby(level=0)['log2FoldChange'].median()

ledger = marker_tbl.copy()
ledger['n_pat_bh'] = SIG_M[:, mcols].sum(axis=0)
ledger['n_hc_bh'] = SIG_HC_M[:, mcols].sum(axis=0)
ledger['n_pat_z196'] = (np.abs(Zm_pat) > MARKER_Z).sum(axis=0)
ledger['n_hc_z196'] = (np.abs(Zm_hc) > MARKER_Z).sum(axis=0)
ledger['group_best_padj'] = ledger['ens'].map(best_padj)
ledger['group_best_rawp'] = ledger['ens'].map(best_rawp)
ledger['group_med_lfc'] = ledger['ens'].map(best_lfc)
ledger['group_sig_any'] = ledger['group_best_padj'].fillna(1.0) < DESEQ2_PADJ
ledger = ledger.sort_values(['n_pat_bh', 'n_pat_z196'], ascending=False)
ledger.to_csv(OUT / 'marker_ledger.csv', index=False)

cols = ['set', 'symbol', 'detect_frac', 'n_pat_bh', 'n_hc_bh', 'n_pat_z196', 'n_hc_z196',
        'group_best_padj', 'group_best_rawp', 'group_sig_any']
print('per-marker: individual side vs the best the group side manages in any of 20 runs')
print(ledger.loc[ledger['detected'] | (ledger['n_pat_bh'] > 0), cols].round(4).to_string(index=False))
nsig = ledger['n_pat_bh'] > 0
print(f'\nmarkers BH-significant in >=1 patient: {int(nsig.sum())}, of which '
      f'{int((nsig & ~ledger["group_sig_any"]).sum())} are padj>=0.05 in every group run')
print(f'same restricted to the detected panel: {int((nsig & ledger["detected"]).sum())}')

## The genes DESeq2 calls and the normative model does not

`deg_orphan` from the matched table above, opened up. Every gene the matched run calls at
`padj<0.05` is split by whether DESeq2's LFC saturated (`|LFC| > SAT_LFC`, the separation bound:
one arm all zero). The saturated orphans fall into two cases, and the figure separates them.


In [ ]:
from scipy.stats import mannwhitneyu

ORPH_DESIGN, ORPH_CASE, ORPH_GRP = RUVG_PICK, 'PancreaticNeoplasm', 'all neoplasm'
SAT_LFC = 20
ORPH_PATH = OUT / 'orphan_diag.csv'

m_orph = group_mask(ORPH_GRP)
D_orph = sorted(deg[(ORPH_CASE, ORPH_DESIGN)])
nm_orph = set(genes[SIG[m_orph].any(axis=0)])
orph_cols = np.array([gpos[g] for g in D_orph])
orph_samples = np.concatenate([names_neo[m_orph], names_hc])
orph_is_pat = np.arange(len(orph_samples)) < int(m_orph.sum())

if ORPH_PATH.exists():
    orph = pd.read_csv(ORPH_PATH)
else:
    Zp, Zh = Z_neo[np.ix_(m_orph, orph_cols)], Z_hc_qc[:, orph_cols]
    r_orph = pd.read_csv(DEG_DIR / f'moore_b1__{ORPH_CASE}' /
                         f'{ORPH_DESIGN}.csv.gz', index_col=0).reindex(D_orph)
    cnt_o, lib_o = raw_counts(orph_samples, D_orph)
    tot_o = cnt_o.sum(axis=0)
    orph = pd.DataFrame(dict(
        ens=D_orph, symbol=[sym_of.get(g, g.split('.')[0]) for g in D_orph],
        orphan=[g not in nm_orph for g in D_orph],
        baseMean=r_orph['baseMean'].values, lfc=r_orph['log2FoldChange'].values,
        padj=r_orph['padj'].values,
        detect_pat=(cnt_o[orph_is_pat] > 0).mean(axis=0),
        detect_hc=(cnt_o[~orph_is_pat] > 0).mean(axis=0),
        n_nonzero=(cnt_o > 0).sum(axis=0), tot_counts=tot_o,
        top1_share=cnt_o.max(axis=0) / np.maximum(tot_o, 1),
        driver_is_hc=[not orph_is_pat[cnt_o[:, k].argmax()] for k in range(cnt_o.shape[1])],
        meanZ_pat=Zp.mean(axis=0), sdZ_pat=Zp.std(axis=0, ddof=1), meanZ_hc=Zh.mean(axis=0),
        maxabsZ_pat=np.abs(Zp).max(axis=0),
        n_z196=(np.abs(Zp) > MARKER_Z).sum(axis=0),
        n_bh=SIG[np.ix_(m_orph, orph_cols)].sum(axis=0),
        mwu_p=[mannwhitneyu(Zp[:, k], Zh[:, k]).pvalue for k in range(Zp.shape[1])]))
    orph['klass'] = np.where(orph['lfc'].abs() > SAT_LFC, 'saturated', 'graded')
    orph.to_csv(ORPH_PATH, index=False)

HC_POOL_PATH = OUT / 'orphan_hc_pool_cpm.npy'

hc_pool = pd.read_csv(config.ZSCORES_MIXED_DIR / 'hc_meta.csv')['sample'].values
if HC_POOL_PATH.exists():
    hc_pool_cpm = np.load(HC_POOL_PATH)
else:
    cnt_p, lib_p = raw_counts(hc_pool, D_orph)
    hc_pool_cpm = cnt_p / lib_p[:, None] * 1e6
    np.save(HC_POOL_PATH, hc_pool_cpm)

orph = orph.set_index('ens').reindex(D_orph).reset_index()
orph['hc_pool_nz'] = (hc_pool_cpm > 0).sum(axis=0)
orph['hc_pool_maxcpm'] = hc_pool_cpm.max(axis=0)
orph['case'] = np.where(~orph['orphan'], 'also NM',
                        np.where(orph['driver_is_hc'], 'HC outlier', 'below threshold'))

print(f'{ORPH_DESIGN} / {ORPH_CASE}: {len(orph)} DE genes, '
      f'{int(orph.orphan.sum())} called by no single patient  '
      f'(patients n={int(m_orph.sum())}, batch-matched HC n={len(names_hc)})')
print(orph.groupby(['klass', 'orphan'])
      .agg(n=('ens', 'size'), top1_share=('top1_share', 'median'),
           n_nonzero=('n_nonzero', 'median'), tot_counts=('tot_counts', 'median'),
           driver_is_hc=('driver_is_hc', 'mean'), maxabsZ_pat=('maxabsZ_pat', 'median'),
           abs_meanZ_pat=('meanZ_pat', lambda s: s.abs().median()),
           n_bh=('n_bh', 'median')).round(3).to_string())
print(f'healthy pool behind the Z: {len(hc_pool)} HC samples')
print(orph[orph['klass'] == 'saturated'].groupby('case')
      .agg(n=('ens', 'size'), top1_share=('top1_share', 'median'),
           n_nonzero=('n_nonzero', 'median'), hc_pool_nz=('hc_pool_nz', 'median'),
           maxabsZ_pat=('maxabsZ_pat', 'median'), n_bh=('n_bh', 'median')).round(2).to_string())

In [ ]:
FS_LG = dict.fromkeys(FS, FS_ANNOT_LG)
C_PATIENT, C_HEALTHY, C_POOL = '#B2182B', '#4D4D4D', '#BDBDBD'
BH_Z = float(norm.isf(FDR_Q / len(genes) / 2))

sat_pos = np.where(orph['klass'].values == 'saturated')[0]
sat = orph.iloc[sat_pos]
cnt_s, lib_s = raw_counts(orph_samples, sat['ens'].tolist())
cpm_s = np.log10(cnt_s / lib_s[:, None] * 1e6 + 1)
pool_s = np.log10(hc_pool_cpm[:, sat_pos] + 1)


def pick(case):
    c = np.where(sat['case'].values == case)[0]
    return c[np.argsort(sat['padj'].values[c])]


def jitter(ax, k, off, v, col, rj):
    ax.plot(k + off + rj.normal(0, 0.045, len(v)), v, '.', ms=2.8, color=col, alpha=0.65)


def gene_axis(ax, cols, extra=None):
    ax.set_xticks(range(len(cols)))
    lab = sat['symbol'].values[cols]
    ax.set_xticklabels(lab if extra is None else [f'{s}\n{e}' for s, e in zip(lab, extra)],
                       rotation=90)
    ax.set_ylabel('log10(CPM + 1)')
    ax.set_xlim(-0.7, len(cols) - 0.3)


c1, c2 = pick('HC outlier'), pick('below threshold')
n_sat_orph = int(sat['orphan'].sum())

with plt.rc_context(FS_LG):
    fig = plt.figure(figsize=(15.0, 9.0))
    gs = fig.add_gridspec(2, 2, width_ratios=[2.05, 1.0], hspace=0.85, wspace=0.2)

    ax = fig.add_subplot(gs[0, :])
    rj = np.random.default_rng(SEED)
    for k, c in enumerate(c1):
        jitter(ax, k, -0.2, cpm_s[orph_is_pat, c], C_PATIENT, rj)
        jitter(ax, k, 0.2, cpm_s[~orph_is_pat, c], C_HEALTHY, rj)
        ax.plot([k - 0.42, k], [cpm_s[orph_is_pat, c].max()] * 2, color=C_PATIENT, lw=1.2,
                zorder=6)
        ax.plot([k, k + 0.42], [cpm_s[~orph_is_pat, c].max()] * 2, color=C_HEALTHY, lw=1.2,
                zorder=6)
    gene_axis(ax, c1)
    ax.set_title('case 1 -- the peak sample is a healthy control  '
                 f'(all {len(c1)} of the {n_sat_orph} saturated genes no patient is called on)',
                 fontsize=FS_ANNOT_LG, pad=26)
    ax.plot([], [], 'o', ms=3, color=C_PATIENT, label=f'patient (n={int(orph_is_pat.sum())})')
    ax.plot([], [], 'o', ms=3, color=C_HEALTHY,
            label=f'HC, this batch (n={int((~orph_is_pat).sum())})')
    ax.plot([], [], '-', color=C_PATIENT, lw=1.2, label='patient maximum')
    ax.plot([], [], '-', color=C_HEALTHY, lw=1.2, label='HC maximum, this batch')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.10), ncol=4, frameon=False,
              handletextpad=0.2, columnspacing=1.6, fontsize=9)

    ax = fig.add_subplot(gs[1, 0])
    rj = np.random.default_rng(SEED)
    pool_nz_lab = []
    for k, c in enumerate(c2):
        nzv = pool_s[pool_s[:, c] > 0, c]
        pool_nz_lab.append(f'{len(nzv)}/{len(hc_pool)}')
        v = ax.violinplot(nzv, positions=[k], widths=1.5, showextrema=False)
        body = v['bodies'][0]
        body.set(facecolor=C_POOL, alpha=0.9, edgecolor='none', zorder=0)
        vt = body.get_paths()[0].vertices
        vt[:, 0] = np.clip(vt[:, 0], k - 0.42, k)
        ax.plot([k - 0.42, k], [nzv.max()] * 2, color='0.35', lw=1.2, zorder=6)
        ax.plot([k, k + 0.42], [cpm_s[orph_is_pat, c].max()] * 2, color=C_PATIENT, lw=1.2,
                zorder=6)
        jitter(ax, k, -0.2, cpm_s[~orph_is_pat, c], C_HEALTHY, rj)
        jitter(ax, k, 0.22, cpm_s[orph_is_pat, c], C_PATIENT, rj)
    gene_axis(ax, c2, pool_nz_lab)
    ax.set_title('case 2 -- the peak is a patient, the healthy pool covers it  '
                 f'(all {len(c2)} of the same {n_sat_orph})', fontsize=FS_ANNOT_LG)
    ax.fill_between([], [], color=C_POOL, alpha=0.9, label='HC pool (NM)')
    ax.plot([], [], ls='', marker='', label=' ')
    ax.plot([], [], '-', color='0.35', lw=1.2, label='HC pool maximum')
    ax.plot([], [], 'o', ms=3, color=C_HEALTHY, label='HC, this batch')
    ax.plot([], [], '-', color=C_PATIENT, lw=1.2, label='patient maximum, this batch')
    ax.plot([], [], 'o', ms=3, color=C_PATIENT, label='patient')
    ax.legend(loc='upper right', ncol=3, frameon=False, handletextpad=0.3, columnspacing=1.2 , fontsize=9)

    ax = fig.add_subplot(gs[1, 1])
    for case, col, mk in (('HC outlier', C_HEALTHY, 'o'), ('below threshold', C_PATIENT, 'D'),
                          ('also NM', '#88A0B8', 'o')):
        sel = sat['case'].values == case
        ax.plot(sat['hc_pool_nz'].values[sel], sat['maxabsZ_pat'].values[sel], mk, ms=4.5,
                color=col, alpha=0.8, label=f'{case} (n={int(sel.sum())})')
    ax.axhline(BH_Z, color='k', lw=0.9, ls='--')
    ax.annotate(f'BH q<{FDR_Q} needs |Z| > {BH_Z:.1f}', (0.03, BH_Z),
                xycoords=('axes fraction', 'data'), textcoords='offset points', xytext=(0, 4),
                fontsize=FS_ANNOT_LG)
    ax.set_xlabel(f'HC of the pool with a read (of {len(hc_pool)})')
    ax.set_ylabel('max |Z| over patients')
    ax.legend(loc='upper right', frameon=False, handletextpad=0.2, fontsize=9)

fig.savefig(FIG / 'fig_orphan_saturated.png', dpi=300, bbox_inches='tight')
print(sat[sat['case'] != 'also NM'].sort_values(['case', 'padj'])[
    ['case', 'symbol', 'lfc', 'padj', 'n_nonzero', 'top1_share', 'hc_pool_nz', 'hc_pool_maxcpm',
     'maxabsZ_pat', 'n_bh']].round(3).to_string(index=False))

In [ ]:
HC_BATCH_MIN = 20
GRADED_PATH = OUT / 'graded_orphan_batchhc.csv'

if GRADED_PATH.exists():
    graded = pd.read_csv(GRADED_PATH)
else:
    graded = orph[(orph['klass'] == 'graded') & orph['orphan']].copy()
    gcols = np.array([gpos[e] for e in graded['ens']])
    hc_meta_all = pd.read_csv(config.ZSCORES_MIXED_DIR / 'hc_meta.csv')
    Z_hc_all = np.load(config.ZSCORES_MIXED_DIR / 'Z_hc_shash.npy')[:, gcols]
    sizes = hc_meta_all.groupby('batch').size()
    hc_batches = sizes[sizes >= HC_BATCH_MIN].index.tolist()
    B = np.vstack([Z_hc_all[(hc_meta_all['batch'] == b).values].mean(axis=0) for b in hc_batches])
    other = B[[i for i, b in enumerate(hc_batches) if b != MATCHED_BATCH]]
    graded['hc_batch_lo'] = B.min(axis=0)
    graded['hc_batch_hi'] = B.max(axis=0)
    graded['hc_batch_sd'] = B.std(axis=0, ddof=1)
    graded['hc_other_absmax'] = np.abs(other).max(axis=0)
    graded['pat_rank_vs_hc'] = (np.abs(other) >= np.abs(graded['meanZ_pat'].values)[None, :]).mean(axis=0)
    graded['patient_specific'] = graded['pat_rank_vs_hc'] == 0
    graded = graded.sort_values('meanZ_pat')
    graded.to_csv(GRADED_PATH, index=False)
    print(f'HC batches with n >= {HC_BATCH_MIN}: {len(hc_batches)} '
          f'(sizes {sizes[hc_batches].min()}-{sizes[hc_batches].max()})')

print(f'graded orphans: {len(graded)}')
print(graded[['meanZ_pat', 'meanZ_hc', 'hc_batch_sd', 'hc_other_absmax',
              'pat_rank_vs_hc']].median().round(3).to_string())
print(f"\npatient |mean Z| exceeded by at least one healthy batch: "
      f"{int((graded.hc_other_absmax >= graded.meanZ_pat.abs()).sum())}/{len(graded)}")
print(f"outside every healthy batch mean: "
      f"{int(graded.patient_specific.sum())}/{len(graded)} "
      f"({', '.join(graded.loc[graded.patient_specific, 'symbol'])})")

## 논문 DEG결과와의 비교 (Github 공개 재현 가능 코드 기반으로 결과 재생성)

Moore et al. 2025의 그룹 DE(CEDAR, PDAC vs 나머지 환자, Wilcoxon)와 우리 NM의 샘플별 편차(BH q<0.05)만 비교한다.
샘플은 Moore CEDAR = 우리 Batch_1 (PDAC 20, IPMN 16, 췌장염 58, HC 48 = Moore의 benign).
Moore DE 표는 저자 코드(`config.MOORE2025_DE_R`)를 공개 데이터에 그대로 돌린 결과이고, CEDAR 29개가 논문과 정확히 일치한다.
BCC 열은 같은 함수를 BCC에 적용한 것으로, 논문은 BCC에서 유전자별 검정을 보고하지 않았다(분류기 AUC만).

### 데이터 불러오기

In [ ]:
import subprocess
from scipy.stats import spearmanr

MOORE_FDR = 0.05
LIVER_UP = ['SERPINA1', 'CRP', 'ACOT12', 'AFMID']
MOORE_DE = {c: config.MOORE2025_DE_DIR / f'moore2025_de_{c}.csv' for c in ('cedar', 'bcc')}
if not all(p.exists() for p in MOORE_DE.values()):
    subprocess.run([config.RSCRIPT, str(config.MOORE2025_DE_R), str(config.MOORE2025_REPO_DIR),
                    str(config.MOORE2025_DE_DIR)], check=True)
mde = (pd.read_csv(MOORE_DE['cedar']).merge(pd.read_csv(MOORE_DE['bcc']), on='genes', suffixes=('_cedar', '_bcc'))
       .rename(columns={'genes': 'symbol'}))
MOORE29 = sorted(mde.loc[mde['FDR_cedar'] < MOORE_FDR, 'symbol'])
assert len(MOORE29) == 29
sym2ensg = pd.read_csv(config.MOORE2025_REPO_DIR / 'g_to_e.csv').drop_duplicates('gene').set_index('gene')['ensemble']
eng_of = {g.split('.')[0]: g for g in genes}
mde['ensg'] = mde['symbol'].map(sym2ensg)
mde['ens'] = mde['ensg'].map(eng_of)
train = pd.read_csv(config.ENGINE_MIXED_DIR / 'training_summary.csv').assign(ensg=lambda d: d['gene'].str.split('.').str[0])
with h5py.File(config.H5AD_PATH, 'r') as h:
    gt = h['var/GeneType']
    gtype = pd.Series(pd.Categorical.from_codes(gt['codes'][()], dec(gt['categories'][()])).astype(str),
                      index=[v.split('.')[0] for v in dec(h['var/_index'][()])])
unmapped = mde.loc[mde['symbol'].isin(MOORE29) & mde['ens'].isna(), ['symbol', 'ensg']].assign(
    gene_type=lambda d: d['ensg'].map(gtype),
    engine=lambda d: d['ensg'].map(train.set_index('ensg')['fail_reason']).fillna('not modeled (protein-coding only)'))

is_pdac, is_ipmn = stage_neo == 'PDAC', stage_neo == 'IPMN'
b1_panc = a_panc & (src['dis_batch'] == MATCHED_BATCH)
GZ = {'HC (matched)': (Z_hc_qc, SIG_HC), 'Pancreatitis': (src['Z_dis'][b1_panc], bh_sig_matrix(src['Z_dis'][b1_panc])),
      'IPMN': (Z_neo[is_ipmn], SIG[is_ipmn]), 'PDAC': (Z_neo[is_pdac], SIG[is_pdac])}
Zp, Sp = GZ['PDAC']
cmap = pd.read_csv(config.GROUP_VS_INDIV_DIR / 'moore_cohort_map.csv')

m29 = mde[mde['symbol'].isin(MOORE29)].copy()
k29 = m29['ens'].map(gpos)
ok = k29.notna().values
k = k29[ok].astype(int).values
m29['PDAC median Z'], m29['HC median Z'], m29['PDAC sig (BH)'] = np.nan, np.nan, np.nan
m29.loc[ok, 'PDAC median Z'] = np.median(Zp[:, k], axis=0)
m29.loc[ok, 'HC median Z'] = np.median(Z_hc_qc[:, k], axis=0)
m29.loc[ok, 'PDAC sig (BH)'] = Sp[:, k].sum(axis=0)
m29['NM'] = np.select([~ok, m29['PDAC sig (BH)'] > 0], ['not in our genes', 'seen in patients'], 'group shift only')

nm_only = pd.DataFrame(dict(ens=genes, symbol=sym_of.reindex(genes).values, n_pdac=Sp.sum(axis=0),
                            n_hc=SIG_HC.sum(axis=0), max_abs_Z=np.abs(Zp).max(axis=0)))
nm_only = nm_only[nm_only['n_pdac'] >= 1].merge(mde[['ens', 'PValue_cedar', 'FDR_cedar']], on='ens', how='left')
nm_only['moore'] = np.select([nm_only['FDR_cedar'] < MOORE_FDR, nm_only['PValue_cedar'].notna()],
                             ['Moore FDR<0.05', 'Moore tested, n.s.'], 'not in Moore table')

lk = [gpos[e] for e in mde.set_index('symbol').loc[LIVER_UP, 'ens']]
liver = pd.concat([pd.DataFrame(dict(sample=src_names, group=g, liver4_meanZ=Z[:, lk].mean(axis=1)))
                   for g, (Z, _), src_names in zip(GZ, GZ.values(),
                                                   [names_hc, src['dis'][b1_panc], names_neo[is_ipmn], names_neo[is_pdac]])])
liver = liver.merge(cmap[['sample', 'Stage', 'Met_Site']], on='sample', how='left')
LCPM_PATH = OUT / 'moore2025_log10cpm.pkl'
ALL_S = np.unique(np.concatenate([src['hc'], src['dis'][b1_panc], names_neo]))


def log_cpm(ens_list):
    tab = pd.read_pickle(LCPM_PATH) if LCPM_PATH.exists() else pd.DataFrame(index=ALL_S)
    new = [e for e in dict.fromkeys(ens_list) if e not in tab.columns]
    if new:
        c, lib = raw_counts(tab.index.values, new)
        tab = pd.concat([tab, pd.DataFrame(np.log10(c / lib[:, None] * 1e6 + 1), index=tab.index, columns=new)],
                        axis=1)
        tab.to_pickle(LCPM_PATH)
    return tab


NOM_Z = float(norm.isf(0.05 / 2))
BH_Z_PDAC = float(np.median([np.abs(z[sg]).min() for z, sg in zip(Zp, Sp) if sg.any()]))
GRP3 = [(f'HC pool, NM reference (n={len(src["hc"])})', src['hc'], src['Z_hc'], '0.82'),
        (f'HC / benign (n={len(names_hc)})', names_hc, Z_hc_qc, '#4D4D4D'),
        (f'PDAC (n={int(is_pdac.sum())})', names_neo[is_pdac], Zp, STAGE_C['PDAC'])]


def violin_or_line(ax, y, x, col):
    if np.ptp(y) == 0:
        ax.hlines(y[0], x - 0.12, x + 0.12, color=col, lw=3)
        return
    vb = ax.violinplot(y, positions=[x], widths=0.25, showextrema=False)
    vb['bodies'][0].set(facecolor=col, edgecolor='none', alpha=1, zorder=0)


def gene_panel_fig(ens_list, ticks, fname, tick_colors=None, xlabel=''):
    lc = log_cpm(ens_list)
    rj = np.random.default_rng(SEED)
    with plt.rc_context(FS):
        fig, (a_c, a_z) = plt.subplots(2, 1, figsize=(0.6 * len(ens_list) + 1.5, 6.6), sharex=True)
        for x, e in enumerate(ens_list):
            k = gpos[e]
            for off, (lab, nm, Z, col) in zip((-0.27, 0.0, 0.27), GRP3):
                v, z = lc.loc[nm, e].values, Z[:, k]
                if off < 0:
                    violin_or_line(a_c, v, x + off, col)
                    violin_or_line(a_z, z, x + off, col)
                    continue
                xs = x + off + rj.normal(0, 0.035, len(v))
                sg = Sp[:, k] if Z is Zp else np.zeros(len(v), bool)
                for ax, y in ((a_c, v), (a_z, z)):
                    ax.plot(xs, y, '.', ms=2.8, color=col, alpha=0.65)
                    ax.scatter(xs[sg], y[sg], s=36, facecolor='none', edgecolor='#D90429', lw=1.2, zorder=4)
                    ax.hlines(np.median(y), x + off - 0.11, x + off + 0.11, color=col, lw=2, zorder=5)
        for y, ls in ((NOM_Z, ':'), (BH_Z_PDAC, '--')):
            for sg_ in (1, -1):
                a_z.axhline(sg_ * y, color='#D90429', ls=ls, lw=0.9)
        a_z.set_xticks(range(len(ens_list)), ticks, rotation=90)
        for t, c in zip(a_z.get_xticklabels(), tick_colors or ['black'] * len(ens_list)):
            t.set_color(c)
        a_z.set_xlim(-0.6, len(ens_list) - 0.4)
        a_c.set_ylabel('log10(CPM + 1)')
        a_z.set_ylabel('Z (NM)')
        a_z.set_xlabel(xlabel)
        handles = [plt.Rectangle((0, 0), 1, 1, color='0.82'),
                   plt.Line2D([], [], ls='', marker='o', ms=3, color='#4D4D4D'),
                   plt.Line2D([], [], ls='', marker='o', ms=3, color=STAGE_C['PDAC']),
                   plt.Line2D([], [], ls='', marker='o', ms=6, mfc='none', mec='#D90429'),
                   plt.Line2D([], [], ls=':', color='#D90429'), plt.Line2D([], [], ls='--', color='#D90429')]
        labels = [g[0] for g in GRP3] + ['BH-significant in that PDAC', 'nominal |Z|>1.96',
                                         f'BH q<0.05, |Z|>{BH_Z_PDAC:.1f} (median PDAC cutoff)']
        fig.legend(handles, labels, frameon=False, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.07))
        fig.tight_layout()
        fig.savefig(FIG / fname, dpi=300, bbox_inches='tight')
        plt.show()


print('Moore 29 not modeled by NM:')
display(unmapped.reset_index(drop=True))
print(f'Moore genes mapped by Ensembl ID: {mde["ens"].notna().sum()}/{len(mde)} | ' 
      f'Moore CEDAR: {len(mde)} genes tested, 29 markers | mapped to our genes: {int(ok.sum())}/29 | '
      f'NM genes with >=1 BH-significant PDAC: {len(nm_only)}')

### 1. Moore 29개 마커: CEDAR → BCC → NM
`NM`: PDAC 20명 중 BH 유의 환자가 있으면 `seen in patients`, 없으면 `group shift only`.

In [ ]:
t1 = m29.assign(**{'BCC same direction': np.sign(m29['LogFC_cedar']) == np.sign(m29['LogFC_bcc'])})
t1 = t1[['symbol', 'LogFC_cedar', 'FDR_cedar', 'BCC same direction', 'PValue_bcc', 'FDR_bcc',
         'PDAC median Z', 'HC median Z', 'PDAC sig (BH)', 'NM']].rename(columns={
    'symbol': 'gene', 'LogFC_cedar': 'CEDAR LFC', 'FDR_cedar': 'CEDAR FDR', 'PValue_bcc': 'BCC p', 'FDR_bcc': 'BCC FDR'})
t1 = t1.sort_values(['NM', 'CEDAR LFC']).reset_index(drop=True)
t1.to_csv(OUT / 'moore2025_markers_vs_nm.csv', index=False)
print(f"BCC: same direction {int(t1['BCC same direction'].sum())}/29, p<0.05 {int((t1['BCC p'] < 0.05).sum())}/29, "
      f"FDR<0.05 {int((t1['BCC FDR'] < 0.05).sum())}/29")
print(t1['NM'].value_counts().to_string())
display(t1.round(3))

### 2. Moore에만 나온 유전자: NM은 왜 안 잡았나
NM에서 BH 유의 PDAC 환자가 0명인 18개를 이유별로 나눈다. Z는 Moore 방향이 위(+)가 되도록 부호를 맞췄다.
- **controls deviate (opposite)**: HC(benign)가 Moore 반대 방향으로 정상에서 벗어나 있고 PDAC는 정상 근처. 그룹 차이는 대조군 몫이다.
- **cohort-wide shift**: HC도 같은 방향으로 벗어나 있고 PDAC는 조금 더 갈 뿐. 코호트 전체의 이동이다.
- **small shift in PDAC**: HC는 정상이고 PDAC 전반이 1 SD 미만으로 이동. 어느 환자도 개인 기준을 넘지 못한다 (NM이 놓치는 경우).
- **sparse gene**: 검출률이 낮아 소수 샘플에만 read가 있다.

기준: HC 중앙 Z가 ±`HC_OFFSET`을 넘으면 HC 이탈, 검출률 < `MIN_DETECT`면 sparse (둘 다 임의 기준).
그림은 3번과 같은 구조: 위 = 발현량(log10 CPM), 아래 = 같은 샘플의 NM Z. 회색 바이올린 = HC 전체 pool(NM 기준),
진한 회색 = 이 코호트 HC(benign), 파랑 = PDAC, 가로선 = 중앙값. Z 축의 점선 = 보정 전(|Z|>1.96), 파선 = BH 기준(PDAC 샘플 중앙값).
유전자는 이유별로 묶었고 라벨 색이 이유를 나타낸다.

In [ ]:
HC_OFFSET = 0.3
MIN_DETECT = 0.3
WHY = ['controls deviate (opposite)', 'cohort-wide shift', 'small shift in PDAC', 'sparse gene']
WHY_C = dict(zip(WHY, ['#4D4D4D', '#E69F00', STAGE_C['PDAC'], '#A6A6A6']))

g2 = m29[m29['NM'] == 'group shift only'].copy()
sgn2 = np.sign(g2['LogFC_cedar'].values)
cnt2, _ = raw_counts(np.concatenate([names_neo[is_pdac], names_hc]), g2['ens'].tolist())
g2['detect_frac'] = (cnt2 > 0).mean(axis=0)
g2['HC Z'] = g2['HC median Z'] * sgn2
g2['PDAC Z'] = g2['PDAC median Z'] * sgn2
g2['why'] = np.select([g2['detect_frac'] < MIN_DETECT, g2['HC Z'] <= -HC_OFFSET, g2['HC Z'] >= HC_OFFSET],
                      [WHY[3], WHY[0], WHY[1]], WHY[2])
g2['why'] = pd.Categorical(g2['why'], WHY, ordered=True)
g2 = g2.sort_values(['why', 'PDAC Z'], ascending=[True, False])
gene_panel_fig(g2['ens'].tolist(), g2['symbol'].tolist(), 'fig_moore_only_why.png',
               tick_colors=[WHY_C[w] for w in g2['why']],
               xlabel='  |  '.join(f'{w} (n={int((g2["why"] == w).sum())})' for w in WHY if (g2['why'] == w).any()))
display(g2[['symbol', 'why', 'LogFC_cedar', 'FDR_cedar', 'detect_frac', 'HC Z', 'PDAC Z']].round(3).reset_index(drop=True))

### 3. NM에서는 나왔는데 Moore DE에서는 나오지 않은 신호 전체

In [ ]:
N_SHOW = 20

miss = nm_only[nm_only['moore'] != 'Moore FDR<0.05'].sort_values(['n_pdac', 'max_abs_Z'], ascending=False)
print(f'NM signals in PDAC (>=1 patient, BH q<0.05): {len(nm_only)} genes')
display(nm_only['moore'].value_counts().to_frame('genes'))
print('genes missed by Moore, by number of PDAC patients carrying them (HC/benign carriers alongside):')
display(miss.groupby('n_pdac').agg(genes=('ens', 'size'), also_in_hc=('n_hc', lambda s: int((s > 0).sum()))).T)


show = miss.head(N_SHOW)
gene_panel_fig(show['ens'].tolist(), [f'{sy}\n{n}/{int(is_pdac.sum())}' for sy, n in zip(show['symbol'], show['n_pdac'])],
               'fig_nm_only_pattern.png', xlabel='gene (PDAC patients BH-significant / 20)')

# Part D -- 환자 한 명이 보내는 장기 신호

각 샘플에서 BH q<0.05로 **상승**한 유전자 중 간/췌장 세포형 마커(HPA, 위 조직 패널)가 몇 개인지 본다.
유전자 universe는 all arm 222 샘플 중 1개 이상에서 검출된 유전자. 환자 hit은 read >= 10.

- 스토리 1: PDAC 일부에서 간세포 유전자 묶음이 한꺼번에 오르고, HC·췌장염에는 그 수준이 없다. 간전이나 CA19-9로는 설명되지 않는다.

In [ ]:
from scipy.stats import fisher_exact

ORG_GROUPS = {**{s: (Z_A[stage_A == s], names_A[stage_A == s]) for s in NEOPLASTIC},
              'Pancreatitis': (Z_panc, src['dis'][a_panc]), 'HC': (Z_hc_A, names_hc_A)}
ORG_CACHE = OUT / 'organ_signal_detect_frac.npy'
if ORG_CACHE.exists():
    det_frac = np.load(ORG_CACHE)
else:
    c_all, _ = raw_counts(np.concatenate([n for _, n in ORG_GROUPS.values()]), list(genes))
    det_frac = (c_all > 0).mean(axis=0)
    np.save(ORG_CACHE, det_frac)
univ = det_frac >= MARKER_MIN_DETECT
ORG_PANELS = {t: np.isin(genes, tissue_tbl.loc[tissue_tbl['set'] == t, 'ens']) & univ for t in ('liver', 'pancreas')}
UP = {g: marker_sig_matrix(Z[:, univ]) & (True if g == 'HC' else read_ok(n)[:, univ]) for g, (Z, n) in ORG_GROUPS.items()}

rows = []
for g, up in UP.items():
    for t, m in ORG_PANELS.items():
        mu = m[univ]
        hit, nsig = up[:, mu].sum(1), up.sum(1)
        rows.append(dict(group=g, n=len(up), panel=t, observed=int(hit.sum()), expected=nsig.sum() * mu.mean(),
                         samples_with_hit=int((hit > 0).sum())))
organ_enr = pd.DataFrame(rows).assign(obs_over_exp=lambda d: d['observed'] / d['expected'])
organ_enr.to_csv(OUT / 'organ_signal_enrichment.csv', index=False)
print(f'universe: {univ.sum()} genes | liver {ORG_PANELS["liver"].sum()}, pancreas {ORG_PANELS["pancreas"].sum()}')
print(organ_enr.round(2).to_string(index=False))

In [ ]:
from scipy.stats import mannwhitneyu, spearmanr

cmi = cmap.set_index('sample')
liv_u = ORG_PANELS['liver'][univ]
liv_ens = genes[univ][liv_u]
Zg = {g: Z[:, univ][:, liv_u] for g, (Z, _) in ORG_GROUPS.items()}
Sg = {g: UP[g][:, liv_u] for g in UP}


hit = np.vstack([Sg['PDAC'], Sg['IPMN']])
gene_keep = np.where(hit.any(0))[0]
gene_keep = gene_keep[np.argsort(-hit[:, gene_keep].sum(0), kind='stable')]

ALL_ORDER = ['Pancreatitis', 'IPMN', 'Islet Cell Tumor', 'PDAC']
ALL_LAB = {'HC': 'HC', 'Pancreatitis': 'Pancreatitis', 'IPMN': 'IPMN', 'Islet Cell Tumor': 'Islet', 'PDAC': 'PDAC'}
cols, grp, met, ca = [], [], [], []
for g in ALL_ORDER:
    sc = Zg[g][:, gene_keep].mean(1)
    for i in np.argsort(-sc):
        s = ORG_GROUPS[g][1][i]
        cols.append(Zg[g][i, gene_keep])
        grp.append(g)
        met.append(str(cmi['Met_Site'].get(s, '')).lower() == 'liver')
        ca.append(cmi['CA19_9'].get(s, np.nan))
MA, grp, met, ca = np.column_stack(cols), np.array(grp), np.array(met), np.array(ca, dtype=float)
score = MA.mean(0)

pdm = grp == 'PDAC'
print(f'liver genes shown: {len(gene_keep)}')
print(pd.DataFrame({g: pd.Series(score[grp == g]).describe()[['count', '25%', '50%', '75%', 'max']]
                    for g in ALL_ORDER}).T.round(2).to_string())
print(f"PDAC liver met vs not: median {np.median(score[pdm & met]):.2f} vs {np.median(score[pdm & ~met]):.2f}, "
      f"MWU p = {mannwhitneyu(score[pdm & met], score[pdm & ~met]).pvalue:.3f}")
m = pdm & ~np.isnan(ca)
r, p = spearmanr(score[m], np.log10(ca[m]))
print(f'PDAC liver score vs log10 CA19-9: rho = {r:+.2f}, p = {p:.3f} (n={int(m.sum())})')

x = np.arange(len(score))
bounds = np.cumsum([np.sum(grp == g) for g in ALL_ORDER])
bar_c = np.where(met, '#B2182B', np.where(pdm, STAGE_C['PDAC'], '0.55'))
with plt.rc_context(FS):
    fig = plt.figure(figsize=(11, 6.2))
    gs = fig.add_gridspec(3, 1, height_ratios=[3, 0.9, 0.9], hspace=0.08)
    a0, a_s, a_ca = fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), fig.add_subplot(gs[2])
    im = a0.imshow(np.clip(MA, -5, 5), cmap='RdBu_r', vmin=-5, vmax=5, aspect='auto', interpolation='nearest')
    a0.set_yticks(range(len(gene_keep)), [sym_of.get(e, e) for e in liv_ens[gene_keep]])
    a0.set_xticks([])
    for g, b0, b1 in zip(ALL_ORDER, np.r_[0, bounds[:-1]], bounds):
        a0.text((b0 + b1) / 2 - 0.5, -0.9, f'{ALL_LAB[g]}\n(n={b1 - b0})', ha='center', va='bottom',
                fontsize=FS_ANNOT_SM)
    fig.colorbar(im, cax=a0.inset_axes([1.005, 0, 0.01, 1]), label='Z')

    a_s.bar(x, score, width=1, color=bar_c)
    a_s.axhline(0, color='0.6', lw=0.6)
    a_s.set_ylabel('Mean Z of\nliver genes')

    ok = ~np.isnan(ca)
    a_ca.bar(x[ok], ca[ok], width=1, color=bar_c[ok])
    a_ca.axhline(37, color='#D90429', ls='--', lw=0.8)
    a_ca.set_yscale('log')
    a_ca.set_ylim(1, 1e4)
    a_ca.set_ylabel('CA19-9\n(U/mL)')
    a_ca.set_xlabel('samples, sorted by mean Z of liver genes within each group')
    for a in (a0, a_s, a_ca):
        for b in bounds[:-1]:
            a.axvline(b - 0.5, color='black', lw=1)
    for a in (a_s, a_ca):
        a.set_xlim(-0.5, len(x) - 0.5)
        a.set_xticks([])
    a_s.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in ('0.55', STAGE_C['PDAC'], '#B2182B')],
               ['other', 'PDAC', 'confirmed liver metastasis'], frameon=False, loc='upper left', ncol=3,
               fontsize=FS_ANNOT_SM)
    fig.savefig(FIG / 'fig_story_liver_signal.png', dpi=300, bbox_inches='tight')
    plt.show()